# 05-02. LOT·강종별 전체 → 확대 → 비교 → 이력

**담당:** 정수진 · **Issue:** #65

14 LOT 모두를 같은 형식으로 읽는다. 각 강종 안에서 전체 관측, 특이 구간, 같은 강종 비교, 정비 전후, 해석이 끝까지 이어진다. 정비 전후 그림은 강종 폴더와 따로 `figures/05_all_data_timeseries_jinny_02/04_정비전후/`에 사건별로 모았다.

| 파일 | 읽는 흐름 |
|---|---|
| [01 자료·품질·시간 지도](05_all_data_timeseries_jinny_01.ipynb) | 자료 범위 → 연간 흐름 → 월별 관측창·정비 → 품질 |
| [02 LOT·강종별 연결 대조](05_all_data_timeseries_jinny_02.ipynb) | 각 LOT 전체 → 자체 신호·강종 대조 통합 확대 → 이력 → 강종 전체 비교 |
| [03 전체 통계·공통 패턴](05_all_data_timeseries_jinny_03.ipynb) | 수준·IQR·시간 변화 → 구역별 응답 → CP·연관성 |
| [04 탐지·구간 근거](05_all_data_timeseries_jinny_04.ipynb) | 기준선·이동 Z·IsolationForest → 후보 위치 → 근거·해석 범위 |

대상은 CAL01의 제공된 14 LOT·22개 센서와 CAL01 정비/같은 WO의 부품교체, CR1 강종변경·정기수리다. 관측 범위는 실제 가동시간이나 소재 입·출측 시간이 아니다. 같은 강종·같은 경과 분은 비교 기준이며 같은 설정값·속도·두께를 보장하지 않는다. CP-Z4·CP-Z4M은 무차원이며, 물리적 정의와 산식은 별도로 확인해야 한다. 시간 겹침·선후관계만으로 고장 원인이나 정비 효과를 확정하지 않는다.

**이 파일에서 바로 보기:** 

| 강종 | LOT 바로 보기 |
|---|---|
| [SGCC](#grade-sgcc) | [240001](#lot-240001) |
| [SPFH590](#grade-spfh590) | [240024](#lot-240024) · [240108](#lot-240108) |
| [SPCE](#grade-spce) | [240037](#lot-240037) · [240121](#lot-240121) · [240133](#lot-240133) |
| [SPCEN](#grade-spcen) | [240038](#lot-240038) · [240114](#lot-240114) |
| [SPHC](#grade-sphc) | [240044](#lot-240044) · [240061](#lot-240061) · [240069](#lot-240069) · [240091](#lot-240091) |
| [DP590](#grade-dp590) | [240058](#lot-240058) |
| [SPCD](#grade-spcd) | [240117](#lot-240117) |

**정비 전후 바로 보기:** 사건 날짜 순서다. 그림 파일은 `figures/05_all_data_timeseries_jinny_02/` 아래 폴더에 있다.

| 정비 사건 | 바로 보기 | 그림 폴더 |
|---|---|---|
| 06/10~06/14 6월 정기수리 | [240058 → 240061 수리 전후](#mnt-0609) | `04_정비전후/01_0610-0614_6월_정기수리/` |
| 07/08 BM | [240069 BM 직전 관측 · CP와 작업 선후](#mnt-0708) | `04_정비전후/02_0708_BM/` |
| 11/05~11/10 11월 정기수리 · 11/13 TBM·CBM | [240114 → 240117 수리 전후](#mnt-1104) · [정비 전·중·후 · CP와 작업 선후 · 경계 확대](#mnt-1113) | `04_정비전후/03_1105-1113_11월_정기수리_TBM_CBM/` |
| 12/28 CBM | [240133 감지·작업·교체](#mnt-1228) | `04_정비전후/04_1228_CBM/` |


## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. TC의 ℃와 OP의 %는 기존 분석의 표시 가정을 유지합니다. 원본에 단위 사전·목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4M은 무차원 기록값으로 원본 값을 그대로 표시합니다.

In [ ]:
from pathlib import Path
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/T-CR1-CAL01_온도.csv").exists() and (p / "notebooks").is_dir())
DATA_DIR = str(PROJECT_ROOT / "data")
FIG_DIR = str(PROJECT_ROOT / "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

## 자료 읽기와 연결

기존 01과 같은 규칙으로 읽습니다. LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "02"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

files = {"온도": TEMP_CSV, "조업": EVENT_CSV, "수리": REPAIR_CSV,
         "설비": EQP_CSV, "부품": PART_CSV, "정비": MAINT_CSV,
         "교체": PARTCHG_CSV}
frames = {name: pd.read_csv(path, encoding=ENC) for name, path in files.items()}
temp, event, repair, eqp = [frames[k] for k in ["온도", "조업", "수리", "설비"]]
part, maint, partchg = [frames[k] for k in ["부품", "정비", "교체"]]
for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = {240001: "#455a64", 240024: "#d55e00", 240037: "#0072b2", 240038: "#0072b2",
             240044: "#0072b2", 240058: "#a23b57", 240061: "#e69f00", 240069: "#7b2cbf",
             240091: "#009e73", 240108: "#0072b2", 240114: "#e69f00", 240117: "#7b2cbf",
             240121: "#009e73", 240133: "#7b2cbf"}
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": "#b52a49",
       "CBM": "#d05c76", "TBM": "#e38592", "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":"#b52a49","CBM":"#007f86","TBM":"#825a30"})

## 그림 표시 규칙

기존 01~04의 그림 규칙을 그대로 씁니다. 짧은 제목, 공정과 컬럼이 보이는 소제목, 축의 대상·단위, 색 범례를 사용하고 TC는 왼쪽·대응 OP는 오른쪽, CP-Z4는 왼쪽·CP-Z4M은 오른쪽에 둡니다. 저장 크기에서 글자 겹침·잘림과 범례의 그래프 침범을 검사합니다.

**추가한 규칙:** 모든 그림 오른쪽 위에 시간축 태그를 붙입니다. 실제 사건·원본 확대는 `TIME AXIS · 실제 측정 날짜·시각`, 여러 LOT 형태 비교는 `TIME AXIS · 관측 시작 후 경과시간[분]`, 시간축이 없는 요약·결과표는 `COMPARISON · 비교 대상`입니다. 실제 시각 원본 확대에는 위쪽 보조축으로 같은 LOT의 경과 분을 함께 적어, 경과시간 비교 그림의 위치를 바로 찾을 수 있게 합니다.

**CP 표기:** CP-Z4와 CP-Z4M은 모두 무차원이다. 그래프 축은 각각 `CP-Z4 기록값`, `CP-Z4M 기록값`으로 표시한다. CP-Z4M의 `1e-10` 또는 `×1e-10`은 눈금 배율이며 단위가 아니다. 계측 산식과 설비·부품 연결은 단위와 별도로 확인한다.

**범례 이름:** 색·선·마커가 뜻하는 자료와 작업을 적는다. 작업 유형·내용·시간은 남기고, 띠 순서나 화면 위치는 이름에 붙이지 않는다. 확대 표시 `(a)~(c)`와 실제 시간의 전·후 구분은 유지한다.

노트북에는 코드가 그린 그림이 바로 표시되고, 같은 그림이 원본 해상도 PNG로 `figures/05_all_data_timeseries_jinny_02/<구분 폴더>/`에 저장된다. 정비 전후 그림은 `04_정비전후/<사건 폴더>/`에 날짜 순서로 저장된다. 계산 코드는 접어 두고 그림·표·해석은 펼쳐 두었다.

In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "온도[℃]"
    if signal in OP:
        return "제어출력[%]"
    if signal == "CP-Z4M":
        return "CP-Z4M 기록값"
    if signal == "CP-Z4":
        return "CP-Z4 기록값"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=20, fontweight="bold", color="#17344d")
    if subtitle:
        fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=10.8, color="#526d80")
    if handles:
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=10.5, handlelength=2.4,
                   columnspacing=1.8, labelspacing=.65)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}") for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.8인치) 둔다.
PAD = .8
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    fig.set_dpi(160)
    layout = compact_explanations(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in fig.legends:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과"})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(160dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display(fig)
    plt.close(fig)

In [ ]:
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

## 상세 시계열 준비

1초 기대 간격의 품질, 1분 집계, 사후 형태 상태(초기 승온·종료 전환·안정 참고), 같은 강종·안정 참고·같은 경과 분 비교, 실제 작업 교집합을 계산한다. 기존 01~08과 같은 코드다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8)
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 무차원이다. 자체 참고 중앙값과 차이를 탐색하며 정상 한계로 해석하지 않는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

In [ ]:
hypotheses = []
def verdict(hypothesis, result, evidence, reason, use, limit="운전 설정·인과·실제 고장 정보 부족", source="기존 03"):
    hypotheses.append({"가설": hypothesis, "판정": result, "검증 근거": evidence, "판정 이유": reason,
                       "08 활용": use, "남은 한계": limit, "출처": source})

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def clock_axis(ax, day):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    style(ax, f"실제 측정 시각[{day:%m/%d} 시:분]")

def clock_top(ax, start, label="같은 LOT 실제 시각[시:분]"):
    """경과 분 축 위에 대상 LOT의 실제 시각을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda m: base + m / 1440, lambda d: (d - base) * 1440))
    top.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    return top

def mean30(k, cols):
    """팀 자료와 같은 집계: 센서별 유효값 30개 이상인 1분 평균."""
    g = raw[k].set_index("MEAS_DT")[cols]
    return g.resample("1min").mean().where(g.resample("1min").count().ge(30))

def sustained_from(series, limit):
    """기준보다 낮은 상태가 관측 끝까지 이어지는 구간의 시작 시각."""
    s = series.dropna(); low = s.lt(limit)
    if not low.iloc[-1]:
        return None
    i = len(low) - 1
    while i > 0 and low.iloc[i - 1]:
        i -= 1
    return low.index[i]

In [ ]:
# ------------------------------------------------------------------------------------------------
# 예정된 변화와 확실히 볼 후보 ①~④ (01·02·03 노트북이 같은 코드로 계산한다)
# ------------------------------------------------------------------------------------------------
# 예정된 변화: 계획수리 직전의 종료 전환(끄면 출력·온도가 함께 내려감), 직후의 초기 승온(켜면 함께 올라감)
relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "예정된 정지"), ("초기 승온", "재가동 승온")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())

# ① 출력과 온도가 반대로 움직인 구간: 출력 0%(≤2%)가 5분 이상인데 온도가 내려가지 않음 / 출력 100%(≥98%)가 5분 이상인데 온도가 오르지 않음
opposite_rows = []
for k in lots.LOT_NO.astype(int):
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9))
        tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on, bad in [("출력 0%인데 온도 안 내려감", op.le(2), lambda d: d >= 0),
                              ("출력 100%인데 온도 안 오름", op.ge(98), lambda d: d <= 0)]:
            for a, b in runs(on.fillna(False), 5):
                change = tc.iloc[min(b, len(tc) - 1)] - tc.iloc[a]
                if bad(change):
                    opposite_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "ZONE": f"Z{z}", "종류": kind,
                                          "START_MIN": a, "END_MIN": b, "길이[분]": b - a, "온도 변화[℃]": change,
                                          "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=a),
                                          "예정된 변화 중": in_expected(k, a)})
opposite = pd.DataFrame(opposite_rows)

# ②·③ 같은 강종 대비 LOT 전체 수준 차이: 안정 참고 1분 중앙값의 LOT 중앙값 - 같은 강종 다른 LOT 중앙값들의 중앙값
level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(states[k].eq("안정 참고") & quality[k][s].ge(.9)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN
overage = level_gap.loc[level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.ge(2.5)]       # ② 과시효 Z7·Z8이 같은 강종보다 2.5℃ 이상 높은 LOT
OVERAGE_LOTS = sorted(overage.LOT.unique())

# ③ 세정 입구 온도(CLN-IN) LOT 중앙값을 정렬해 가장 큰 간격에서 두 묶음으로 나눈다.
cln = level_gap.loc[level_gap.SENSOR.eq("CLN-IN")].set_index("LOT").LOT_MEDIAN.sort_values()
cut = cln.diff().idxmax()                                   # 큰 묶음의 첫 LOT
CLN_SPLIT = (cln.loc[:cut].iloc[-2] + cln.loc[cut]) / 2
cln_group = pd.Series(np.where(cln.ge(CLN_SPLIT), f"{CLN_SPLIT:.1f}℃ 이상", f"{CLN_SPLIT:.1f}℃ 미만"), index=cln.index)

# ④ CP 변화: 자체 안정 참고 기준과 가장 크게 벗어난 펄스 (예정된 정지 중의 CP 하강은 제외)
cp_main = cp_candidates.loc[[not in_expected(int(r.LOT), r.PEAK_MIN) for r in cp_candidates.itertuples()]]
CP_TOP = cp_main.loc[cp_main.DELTA.abs().idxmax()]

# ① 대표 사례: 출력 0%가 5분 이상인데 온도가 가장 크게 오른 곳
z0 = opposite.loc[opposite["종류"].str.startswith("출력 0%") & ~opposite["예정된 변화 중"]]
Z3_TOP = z0.loc[z0["온도 변화[℃]"].idxmax()]

# 03에서 자세히 본 LOT과 이유 (그림에서 빨간색으로 표시한다)
SELECTED = {}
for k in OVERAGE_LOTS:
    SELECTED.setdefault(int(k), []).append("② Z7·Z8")
SELECTED.setdefault(int(CP_TOP.LOT), []).append("④ CP")
SELECTED.setdefault(int(Z3_TOP.LOT), []).append("① Z3")
SELECT_COLOR = "#d62728"

MARK = {"①": "#c0392b", "②": "#8e44ad", "③": "#16a085", "④": "#d35400"}
MARK_NAME = {"①": "출력·온도 반대 움직임", "②": "과시효 Z7·Z8 수준 차이", "③": "세정 입구 온도 두 수준", "④": "가장 큰 CP 변화"}
EXPECTED_COLOR = "#8d99a6"

In [ ]:
from matplotlib.patches import Rectangle
GRADE_COLOR={"SGCC":"#455a64","SPFH590":"#d55e00","SPCE":"#0072b2","SPCEN":"#e69f00","SPHC":"#009e73","DP590":"#a23b57","SPCD":"#7b2cbf","SPCC":"#8c6d31"}
ZONE_COLOR={1:"#455a64",2:"#d55e00",3:"#0072b2",4:"#e69f00",5:"#009e73",6:"#cc79a7",7:"#8c6d31",8:"#7b2cbf",9:"#56b4e9"}
SENSOR_ZONE_COLOR=dict(ZONE_COLOR)
ZONES=[3,4,5,6]
SELECT_COLOR="#c83745"
MARK_NAME["①"]="출력·온도 응답 확인"
PROCESS["TC-Z9"]="Z9"
def sensor_color(s):
    if s.startswith(("TC-Z","OP-Z")):return SENSOR_ZONE_COLOR[int(s.rsplit('Z',1)[1])]
    return {"CLN-IN":"#455a64","CP-Z4":"#b04482","CP-Z4M":"#7b2cbf","TC-OUT1":"#009e73","TC-OUT2":"#d55e00"}[s]
def sensor_scale(s):return 1e10 if s=="CP-Z4M" else 1
def shown_unit(s):return "CP-Z4M 기록값[×1e-10]" if s=="CP-Z4M" else unit(s)
def when(k,a,b):
    start=lot_meta.loc[k,"START"]
    return f"{start+pd.Timedelta(minutes=float(a)):%m/%d %H:%M:%S}–{start+pd.Timedelta(minutes=float(b)):%m/%d %H:%M:%S}"
def clock_axis(ax,day=None):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    style(ax,"실제 측정 날짜·시각[월/일 시:분]")
def raw_window(lot,a,b):
    r=lot_meta.loc[lot]
    times=pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)),min(r.END,r.START+pd.Timedelta(seconds=int(b*60))),freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)
def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"
def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]
def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")

# 모든 후보를 목록에 남기고, 긴 수준 차이와 짧은 변화를 구분해 표시한다.
review_rows=[]
for e in episodes.itertuples():
    level=level_gap.loc[level_gap.LOT.eq(e.LOT)&level_gap.SENSOR.eq(e.SENSOR),"GAP"].iloc[0]
    long=bool(e.DURATION_MIN>=30)
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":e.SENSOR,"KIND":"같은 강종 차이","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"최대 {e.MAX_ABS_DELTA_C:.2f}℃ · 대조 {e.MIN_PEERS} LOT 이상","LOCAL":not long})
for e in opposite.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":f"TC-{e.ZONE}","KIND":"출력·온도 응답","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"{e.종류} · TC 변화 {e._8:+.2f}℃","LOCAL":True})
for e in cp_candidates.itertuples():
    review_rows.append({"LOT":int(e.LOT),"GRADE":e.GRADE,"SENSOR":"CP-Z4","KIND":"CP 변화","A":float(e.START_MIN),"B":float(e.END_MIN),"DETAIL":f"자체 참고 대비 {e.DELTA:+.3f} · 극값 {e.PEAK_VALUE:.4g}","LOCAL":e.PULSE_WIDTH_MIN<30})
review_catalogue=pd.DataFrame(review_rows).sort_values(["LOT","A","SENSOR"]).reset_index(drop=True)
review_catalogue["REVIEW_ID"]=[f"R{i+1:03d}" for i in range(len(review_catalogue))]
review_catalogue["START"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.A,unit="min")
review_catalogue["END"]=review_catalogue.LOT.map(lot_meta.START)+pd.to_timedelta(review_catalogue.B,unit="min")

def lot_windows(k):
    """겹친 국소 변화의 문맥을 합친다. 긴 수준 차이는 전체 파형·비교 표에서 확인한다."""
    rows=review_catalogue.loc[review_catalogue.LOT.eq(k)&review_catalogue.LOCAL]
    result=[]
    for r in rows.sort_values('A').itertuples():
        a=max(0,r.A-5);b=min(lot_meta.loc[k,"MINUTES"],r.B+5)
        if result and a<=result[-1]["B"]+3:
            result[-1]["B"]=max(result[-1]["B"],b);result[-1]["IDS"].append(r.REVIEW_ID);result[-1]["SENSORS"].add(r.SENSOR)
        else:result.append({"A":a,"B":b,"IDS":[r.REVIEW_ID],"SENSORS":{r.SENSOR}})
    for i,r in enumerate(result):r["WINDOW_ID"]=f"{k}-W{i+1:02d}"
    return result
overview_coverage={};window_coverage={};maintenance_stage_rows=[];window_audit=[]

**그림 저장 이름:** `LOT_월일_시작시각_핵심내용.png`처럼 짧게 적는다.
예: `240069_0708_0225_CP상승.png`. 전체 관측은 `240069_0707_전체관측.png`,
같은 구간의 강종 대조는 해당 LOT 확대 그림에 함께 넣는다. 여러 변화가 겹치면 대표 내용을 이름에 적고,
전체 시간 범위와 함께 본 센서는 그림 설명·파일목록에 남긴다.

월일·시작시각은 **대상 LOT의 확대 창 시작**이며 극값 시각과 다를 수 있다.
모든 자료는 2024년 기록이다. 각 그림 아래 파일 링크와 `그림_파일목록.csv`에서 찾을 수 있다.

## 읽는 순서와 표시 기준

전체 관측은 1분 중앙값과 원본 최소–최대다. 확대는 1초 원본이며 기록 공백을 보간하지 않는다. R 번호는 전수 후보, W 번호는 겹친 국소 후보를 합친 확대창이다. 긴 수준 차이는 전체 파형·동일 강종 비교로 확인하고 표에 보존한다. 후보는 고장 정답이 아니다.

- [SGCC](#grade-sgcc) · [SPFH590](#grade-spfh590) · [SPCE](#grade-spce) · [SPCEN](#grade-spcen) · [SPHC](#grade-sphc) · [DP590](#grade-dp590) · [SPCD](#grade-spcd)

**온도 차이 그림:** 같은 강종이 2 LOT이면 A−B 차이 한 선과 공통 비교 구간을 읽는다. 3 LOT 이상이면 대상에서 다른 대조가 2개 이상인 구간은 실선, 1개인 구간은 보이는 점선으로 구분한다. 대조 1 LOT은 비교 불가와 다르며 기존 다수 대조 후보 규칙은 유지한다.

**확대 그림 통합:** 같은 LOT·같은 확대 창의 자체 신호와 동일 강종 원본은 한 장에 이어 놓는다. 위는 실제 시각, 아래 강종 대조는 경과시간이며 각 축에 구분해서 적는다. `03_강종`에는 강종 전체 비교 그림만 저장한다.

**비교 종류의 색 구분:** 파란 제목 영역은 공정별 비교(선 색=구역), 보라 제목 영역은 강종별 LOT 비교(선 색=LOT 번호)다. 범례는 각 비교 바로 위에 따로 둔다.

In [ ]:
# 저장 이름은 LOT 1회 + 실제 확대 날짜·시각 + 센서·관측 내용으로 만든다.
# WINDOW_ID와 REVIEW_ID는 내부 추적용이며 저장 이름과 그림 제목에는 쓰지 않는다.
image_name_manifest = []

def window_description(k, win):
    rows = review_catalogue.loc[review_catalogue.REVIEW_ID.isin(win['IDS'])]
    kinds = set(rows.KIND)
    labels = []
    if 'CP 변화' in kinds:
        delta = cp_candidates.loc[cp_candidates.LOT.eq(k), 'DELTA'].iloc[0]
        labels.append('CP상승' if delta > 0 else 'CP하강')
    if '출력·온도 응답' in kinds:
        labels.append('출력온도응답')
    if '같은 강종 차이' in kinds:
        labels.append('동일강종차이')
    return '_'.join(labels) or '신호변화'

def window_signal_name(win):
    names = []
    sensors = set(win['SENSORS'])
    for prefix in ['TC-Z', 'OP-Z']:
        zs = sorted(int(s[len(prefix):]) for s in sensors if s.startswith(prefix))
        if zs:
            names.append(prefix + ('-Z'.join([str(zs[0]), str(zs[-1])])
                         if len(zs) > 1 and zs == list(range(zs[0], zs[-1]+1))
                         else '+'.join(map(str, zs))))
    names += sorted(sensors - {s for s in sensors if s.startswith(('TC-Z','OP-Z'))})
    return '_'.join(names)

def image_focus(signals, description):
    if 'CP상승' in description: return 'CP상승'
    if 'CP하강' in description: return 'CP하강'
    if '출력온도응답' in description: return '출력온도'
    if signals == 'CLN-IN': return '세정온도변화'
    if signals == 'TC-Z3': return 'Z3온도차이'
    if signals == 'TC-Z4': return 'Z4온도차이'
    if signals == 'TC-Z7-Z8': return '과시효온도차이'
    if signals.startswith('TC-Z3'): return '가열온도차이'
    return '온도차이'

def lot_image_name(k):
    r = lot_meta.loc[k]
    return f"02_LOT/{r.GRADE}/{k}_{r.START:%m%d}_전체관측.png"

def window_image_name(k, win, peers=False):
    folder = f"03_강종/{lot_meta.loc[k,'GRADE']}" if peers else f"02_LOT/{lot_meta.loc[k,'GRADE']}"
    start = lot_meta.loc[k,'START'] + pd.Timedelta(seconds=round(win['A']*60))
    focus = image_focus(window_signal_name(win),window_description(k,win))
    suffix = '_강종대조' if peers else ''
    return f"{folder}/{k}_{start:%m%d_%H%M}_{focus}{suffix}.png"

def register_image_name(k, win, name, peers=False):
    old = (f"03_강종/{lot_meta.loc[k,'GRADE']}/{win['WINDOW_ID']}_동일강종원본.png" if peers else
           f"02_LOT/{lot_meta.loc[k,'GRADE']}/{k}_{win['WINDOW_ID']}_원본.png")
    image_name_manifest.append({'LOT':k,'모드':'자체 신호·강종 대조' if window_has_peer_comparison(k,win) else '동시 신호 확대',
        '시작':lot_meta.loc[k,'START']+pd.Timedelta(minutes=win['A']),
        '끝':min(lot_meta.loc[k,'END'],lot_meta.loc[k,'START']+pd.Timedelta(minutes=win['B'])),
        '확인 내용':window_description(k,win),'센서':window_signal_name(win),
        '파일':name,'이전 파일':old,'근거 ID':','.join(win['IDS'])})

LOT_PANELS=[("세정 입구",["CLN-IN"],None),
 ("건조",["TC-Z1","TC-Z2"],["OP-Z1","OP-Z2"]),
 ("가열·균열",HEAT,[f"OP-Z{z}" for z in range(3,7)]),
 ("CP",["CP-Z4"],["CP-Z4M"]),
 ("과시효",["TC-Z7","TC-Z8"],["OP-Z7","OP-Z8"]),
 ("Z9",["TC-Z9"],None),
 ("냉각",["TC-OUT1","TC-OUT2"],None)]

def process_axes(ax,k,sensors,a=None,b=None,clock=True,raw_values=False,envelope=True):
    r=lot_meta.loc[k]
    for s in sensors:
        scale=sensor_scale(s)
        if raw_values:
            w=raw_window(k,a,b);x=w.index if clock else (w.index-r.START).total_seconds()/60;y=w[s]*scale
            ax.plot(x,y,color=sensor_color(s),lw=1.3)
        else:
            p=profile[k];x=r.START+pd.to_timedelta(p.index,unit="min") if clock else p.index
            ax.plot(x,p[(s,'median')]*scale,color=sensor_color(s),lw=1.4)
            if envelope:ax.fill_between(x,p[(s,'min')]*scale,p[(s,'max')]*scale,color=sensor_color(s),alpha=.10,lw=0)
    ax.set_ylabel(shown_unit(sensors[0]),labelpad=8)
    if clock:clock_axis(ax)
    else:style(ax)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(labelsize=12)

def full_lot(k,name=None):
    name = lot_image_name(k) if name is None else name
    r=lot_meta.loc[k]; windows=lot_windows(k);seen=[]
    fig=plt.figure(figsize=(28,28))
    gs=fig.add_gridspec(7,2,left=.085,right=.975,top=.89,bottom=.035,hspace=.72,wspace=.20)
    axes=[]
    for row,(title,left,right) in enumerate(LOT_PANELS):
        for col,sensors in [(0,left),(1,right)] if right else [(0,left)]:
            ax=fig.add_subplot(gs[row,col] if right else gs[row,:]);axes.append(ax)
            process_axes(ax,k,sensors);seen.extend(sensors)
            for t in linked_work(k).itertuples():ax.axvspan(t.STRT_DT,t.END_DT,color=COL[t.MNT_TYPE],alpha=.07,lw=0)
            for win in windows:
                if set(sensors)&win['SENSORS'] or set(sensors)&{s.replace('TC','OP') for s in win['SENSORS']}:
                    x0=r.START+pd.Timedelta(minutes=win['A']);x1=r.START+pd.Timedelta(minutes=win['B'])
                    ax.axvspan(x0,x1,color=COL['FOCUS'],alpha=.07,lw=0)
            ax.set_xlim(r.START,r.END)
            ax.set_title(f"{title} [{' · '.join(sensors)}]",loc="left",fontsize=15,pad=13)
    handles=[Line2D([0],[0],color=SENSOR_ZONE_COLOR[z],lw=2,label=f"Z{z}") for z in range(1,10)]
    handles += [Patch(facecolor=COL['FOCUS'],alpha=.2,label="다음 원본 확대 위치"),Patch(facecolor=COL['WORK'],alpha=.2,label="실제 작업 교집합")]
    header(fig,f"{k} · {r.GRADE} 전체 관측",f"{r.START:%m/%d %H:%M:%S}–{r.END:%m/%d %H:%M:%S} · 선: 1분 중앙값 · 같은 색 음영: 원본 최소–최대 · 22개 신호",handles,ncol=6)
    compact_header(fig,.952,.930,title_size=25,sub_size=13,legend_size=12)
    axis_tag(fig,TAG_REAL)
    image_name_manifest.append({'LOT':k,'모드':'전체 관측','시작':r.START,'끝':r.END,
        '확인 내용':'22개 신호 전체 관측','센서':'22개 신호','파일':name,
        '이전 파일':f"02_LOT/{r.GRADE}/{k}_01_전체공정.png",'근거 ID':''})
    finish(fig,name);overview_coverage[k]=set(seen)
    table(review_catalogue.loc[review_catalogue.LOT.eq(k),['REVIEW_ID','SENSOR','KIND','START','END','DETAIL','LOCAL']].rename(columns={'LOCAL':'바로 아래 국소 확대'}))
    note("**전체에서 확인할 것:** 먼저 세정부터 냉각까지 동반 변화와 개별 구역의 움직임을 본다. 붉은 시간창은 다음 원본 확대의 위치다. 긴 수준 차이는 전체 파형과 이어지는 동일 강종 비교에서 확인한다. 시간 공백은 채우지 않았고, CP-Z4M 축의 ×1e-10은 표시 배율이다.")

def _window_detail_single(k,win,name=None,peers=False):
    name = window_image_name(k,win,peers) if name is None else name
    register_image_name(k,win,name,peers)
    r=lot_meta.loc[k];a,b=win['A'],win['B'];sensors=win['SENSORS'];rows=review_catalogue.loc[review_catalogue.REVIEW_ID.isin(win['IDS'])]
    primary=next((s for s in ['TC-Z3','CP-Z4','CLN-IN'] if s in sensors),sorted(sensors)[0])
    pairs=[]
    if set(sensors)&set(HEAT+['CP-Z4']):pairs += [('가열·균열',HEAT,[f'OP-Z{z}' for z in range(3,7)]),('CP',['CP-Z4'],['CP-Z4M'])]
    other=[s for s in sensors if s not in set(HEAT+['CP-Z4'])]
    for s in sorted(other):
        paired=s.replace('TC-','OP-')
        pairs.append((PROCESS[s],[s],[paired] if paired in OP else None))
    fig=plt.figure(figsize=(28,6+4.6*len(pairs)))
    gs=fig.add_gridspec(1+len(pairs),2,height_ratios=[.65]+[1]*len(pairs),left=.09,right=.975,top=.80 if len(pairs)<3 else .86,bottom=.06,hspace=.82,wspace=.20)
    locator=fig.add_subplot(gs[0,:]);p=profile[k]
    x=r.START+pd.to_timedelta(p.index,unit='min') if not peers else p.index
    locator.plot(x,p[(primary,'median')]*sensor_scale(primary),color=LOT_COLOR[k],lw=1.4)
    x0=r.START+pd.Timedelta(minutes=a) if not peers else a;x1=r.START+pd.Timedelta(minutes=b) if not peers else b
    locator.axvspan(x0,x1,color=COL['FOCUS'],alpha=.13,lw=0);locator.axvspan(x0,x1,fill=False,ec=COL['FOCUS'],lw=1.5)
    locator.set_ylabel(shown_unit(primary));locator.set_title(f"(a) {k} 전체 [{primary}] · 상자 = 아래 확대 위치",loc='left',fontsize=15,pad=12)
    clock_axis(locator) if not peers else style(locator)
    for row,(group,left,right) in enumerate(pairs,1):
        for col,ss in [(0,left),(1,right)] if right else [(0,left)]:
            ax=fig.add_subplot(gs[row,col] if right else gs[row,:])
            if peers:
                ids=lots.loc[lots.GRADE.eq(r.GRADE),'LOT_NO'].astype(int).tolist()
                for kk in ids:
                    if lot_meta.loc[kk,'MINUTES']<=a:continue
                    for s in ss:line(ax,kk,s,start=a,end=min(b,lot_meta.loc[kk,'MINUTES']),raw_values=True)
                n=comparison[k,left[0]].N_PEERS.loc[int(a):int(b)]
                for aa,bb in runs(n.lt(2)):ax.axvspan(aa,bb,color='#e9edf0',alpha=.55,lw=0)
                ax.set_xlim(a,b);clock_top(ax,r.START,label=f"대상 {k} 실제 시각[시:분]");style(ax)
            else:
                process_axes(ax,k,ss,a,b,raw_values=True,envelope=False);ax.set_xlim(x0,x1)
                for t in linked_work(k).itertuples():
                    if t.STRT_DT<x1 and t.END_DT>x0:
                        ax.axvspan(max(t.STRT_DT,x0),min(t.END_DT,x1),color=COL[t.MNT_TYPE],alpha=.08,lw=0)
                    for field,style_ in [('DETC_DT',':'),('STRT_DT','--'),('END_DT','-.')]:
                        at=getattr(t,field)
                        if x0<=at<=x1:ax.axvline(at,color=COL['DETECT'] if field=='DETC_DT' else COL[t.MNT_TYPE],ls=style_,lw=1.4)
                elapsed_top(ax,r.START)
            ax.set_title(f"({chr(98+(row-1)*2+col)}) {group} [{' · '.join(ss)}] · 1초 원본",loc='left',fontsize=15,pad=42)
    handles=lot_handles(lots.loc[lots.GRADE.eq(r.GRADE),'LOT_NO']) if peers else [Line2D([0],[0],color=sensor_color(s),lw=2,label=s) for s in dict.fromkeys(s for _,l,rr in pairs for s in l+(rr or []))]
    header(fig,f"{k} · {window_description(k,win).replace('_',' · ')} · {'같은 강종 대조' if peers else '동시 신호 확대'}",f"{r.GRADE} · {when(k,a,b)} · 근거 {', '.join(win['IDS'])}",handles,ncol=6)
    compact_header(fig,.952 if len(pairs)>=3 else .94,.930 if len(pairs)>=3 else .90,title_size=24,sub_size=12,legend_size=11.5);axis_tag(fig,TAG_ELAPSED if peers else TAG_REAL)
    finish(fig,name)
    for rr in rows.itertuples():
        g=raw[k].loc[raw[k].ELAPSED.ge(rr.A)&raw[k].ELAPSED.lt(rr.B)]
        note(f"**{rr.REVIEW_ID} · {rr.SENSOR}:** {rr.DETAIL}. 실제 {rr.START:%m/%d %H:%M:%S}–{rr.END:%m/%d %H:%M:%S}, 원본 유효 기록 {g[rr.SENSOR].count()}개, 범위 {g[rr.SENSOR].min():.4g}~{g[rr.SENSOR].max():.4g}. "+('다른 LOT은 같은 강종의 같은 경과시간이며 같은 실제 날짜가 아니다.' if peers else '같은 실제 시각의 다른 구역과 출력을 함께 읽는다.'))
    window_coverage.setdefault(k,set()).update(win['IDS']);window_audit.append({'LOT':k,'WINDOW':win['WINDOW_ID'],'FILE':name,'MODE':'PEERS' if peers else 'CLOCK','REVIEW_IDS':win['IDS'],'A':a,'B':b})



def window_has_peer_comparison(k, win):
    rows = review_catalogue.loc[review_catalogue.REVIEW_ID.isin(win['IDS'])]
    return bool(rows.KIND.eq('같은 강종 차이').any())

# 비교 조건은 별도의 범위 그림 대신 해당 확대 패널 바로 옆에 짧게 표시한다.
peer_condition_audit = []

def peer_condition(k, sensor, a, b):
    """확대 시간에서 대상·대조의 안정 참고 및 기록량 조건을 함께 확인한다."""
    grid = pd.RangeIndex(int(np.floor(a)), int(np.ceil(b)))
    d = comparison[k, sensor].reindex(grid)
    weights = pd.Series(np.maximum(0, np.minimum(grid + 1, b) - np.maximum(grid, a)), index=grid)
    usable = d.VALUE.notna() & d.DELTA.notna()
    masks = [usable & d.N_PEERS.ge(2), usable & d.N_PEERS.eq(1), ~usable]
    minutes = [float(weights.where(mask, 0).sum()) for mask in masks]
    total = float(weights.sum())
    assert np.isclose(sum(minutes), b - a), '비교 조건이 확대 시간 전체를 포함하지 않음'
    if np.isclose(minutes[0], total):
        counts = d.loc[masks[0], 'N_PEERS']
        lo, hi = int(counts.min()), int(counts.max())
        label = f'대조 {lo} LOT' if lo == hi else f'대조 {lo}~{hi} LOT'
    elif np.isclose(minutes[1], total):
        label = '대조 1 LOT · 참고'
    elif np.isclose(minutes[2], total):
        label = '비교 조건 미충족'
    else:
        parts = []
        # 시간 비율을 정수로 표시하되 반올림한 합계도 100%가 되게 한다.
        percentages = np.asarray(minutes) / total * 100
        rounded = np.floor(percentages).astype(int)
        order = np.argsort(-(percentages - rounded), kind='stable')
        rounded[order[:100 - int(rounded.sum())]] += 1
        assert rounded.sum() == 100
        for text, duration, percent in zip(['2+ LOT', '1 LOT', '불가'], minutes, rounded):
            if duration <= 1e-8:
                continue
            shown = '<1%' if percent == 0 else f'{percent}%'
            parts.append(f'{text} {shown}')
        label = '대조: ' + ' · '.join(parts)
    return {'LOT': int(k), 'SENSOR': sensor, 'A': float(a), 'B': float(b),
            'LABEL': label, 'TWO_OR_MORE_MIN': minutes[0], 'ONE_MIN': minutes[1],
            'UNAVAILABLE_MIN': minutes[2], 'TOTAL_MIN': total}

def mark_peer_condition(fig, ax, k, sensor, a, b):
    condition = peer_condition(k, sensor, a, b)
    peer_condition_audit.append(condition)
    p = ax.get_position()
    limited = condition['ONE_MIN'] > 1e-8 or condition['UNAVAILABLE_MIN'] > 1e-8
    fig.text(p.x1, p.y1 + 42 / 72 / fig.get_figheight(), condition['LABEL'],
             ha='right', va='baseline', fontsize=13, color='#526d80',
             bbox=dict(boxstyle='round,pad=.25', fc='#fff4df' if limited else '#eef3f7',
                       ec='#d0b77f' if limited else '#b9cbd8', lw=.8))

combined_window_audit = []

def window_detail(k, win, name=None, peers=False):
    """같은 확대 창의 대상 신호와 강종 대조를 한 그림 안에서 이어서 읽는다."""
    if not window_has_peer_comparison(k,win):
        return _window_detail_single(k,win,name=name,peers=False)
    name = window_image_name(k,win,False) if name is None else name
    register_image_name(k,win,name,False)
    r=lot_meta.loc[k];a,b=win['A'],win['B'];sensors=set(win['SENSORS'])
    evidence=review_catalogue.loc[review_catalogue.REVIEW_ID.isin(win['IDS'])]
    primary=next((s for s in ['TC-Z3','CP-Z4','CLN-IN'] if s in sensors),sorted(sensors)[0])
    own_pairs=[]
    if sensors & set(HEAT+['CP-Z4']):
        own_pairs += [('가열·균열',HEAT,[f'OP-Z{z}' for z in range(3,7)]),('CP',['CP-Z4'],['CP-Z4M'])]
    for s in sorted(sensors-set(HEAT+['CP-Z4'])):
        paired=s.replace('TC-','OP-')
        own_pairs.append((PROCESS[s],[s],[paired] if paired in OP else None))
    # 강종 대조는 후보 구역별로 나눠 LOT 색이 어떤 센서인지 분명히 한다.
    # 동시 CP 움직임은 바로 위의 대상 LOT CP와 나란히 확인한다.
    peer_pairs=[]
    for group,left,right in own_pairs:
        selected=left if group=='CP' else [s for s in left if s in sensors]
        for s in selected:
            paired='CP-Z4M' if s=='CP-Z4' else s.replace('TC-','OP-')
            peer_pairs.append((group,[s],[paired] if paired in SIGNALS and paired!=s else None))
    count=len(own_pairs)+len(peer_pairs);height=6+4.4*count
    fig=plt.figure(figsize=(28,height))
    gs=fig.add_gridspec(3+count,2,height_ratios=[.65,.65]+[1]*len(own_pairs)+[.65]+[1]*len(peer_pairs),
        left=.09,right=.975,top=1-2.25/height,bottom=.055,hspace=.65,wspace=.22)
    locator=fig.add_subplot(gs[0,:]);p=profile[k]
    locator.plot(r.START+pd.to_timedelta(p.index,unit='min'),p[(primary,'median')]*sensor_scale(primary),color='#455a64',lw=1.5)
    t0=r.START+pd.Timedelta(minutes=a);t1=r.START+pd.Timedelta(minutes=b)
    locator.axvspan(t0,t1,color=COL['FOCUS'],alpha=.13,lw=0)
    locator.axvspan(t0,t1,fill=False,ec=COL['FOCUS'],lw=1.5)
    locator.set_xlim(r.START,r.END);clock_axis(locator);locator.set_ylabel(shown_unit(primary))
    locator.set_title(f'(a) 대상 {k} 전체 [{primary}] · 상자 = 아래 두 대조의 공통 확대 위치',loc='left',fontsize=15,pad=13)
    zone_handles=[Line2D([0],[0],color=sensor_color(s),lw=2,label=(f"대상 Z{s.rsplit('Z',1)[1]} 온도·출력" if s.startswith('TC-Z') else f'대상 {s}'))
        for s in dict.fromkeys(s for _,left,_ in own_pairs for s in left)]
    ids=lots.loc[lots.GRADE.eq(r.GRADE),'LOT_NO'].astype(int).tolist()
    if any(group=='CP' for group,_,_ in own_pairs):
        zone_handles.append(Line2D([0],[0],color=sensor_color('CP-Z4M'),lw=2,label='대상 CP-Z4M'))
    peer_handles=[Line2D([0],[0],color=LOT_COLOR[kk],lw=2.4 if kk==k else 1.3,
        label=f"{kk} · {lot_meta.loc[kk,'START']:%m/%d}"+(' · 대상' if kk==k else '')) for kk in ids]
    peer_handles.append(Patch(facecolor='#e9edf0',label='비교 조건 미충족'))
    observed=set();compared=set();axis_modes=[]
    for row,(group,left,right) in enumerate(own_pairs,2):
        for col,ss in [(0,left),(1,right)] if right else [(0,left)]:
            ax=fig.add_subplot(gs[row,col] if right else gs[row,:])
            ax.set_facecolor('#f7fbff')
            process_axes(ax,k,ss,a,b,raw_values=True,envelope=False);ax.set_xlim(t0,t1);observed.update(ss)
            for work in linked_work(k).itertuples():
                if work.STRT_DT<t1 and work.END_DT>t0:
                    ax.axvspan(max(work.STRT_DT,t0),min(work.END_DT,t1),color=COL[work.MNT_TYPE],alpha=.08,lw=0)
                for field,ls in [('DETC_DT',':'),('STRT_DT','--'),('END_DT','-.')]:
                    stamp=getattr(work,field)
                    if t0<=stamp<=t1:ax.axvline(stamp,color=COL['DETECT'] if field=='DETC_DT' else COL[work.MNT_TYPE],ls=ls,lw=1.4)
            elapsed_top(ax,r.START)
            ax.set_title(f"대상 LOT · {group} [{' · '.join(ss)}] · 실제 측정 시각",loc='left',fontsize=15,pad=42)
            axis_modes.append('CLOCK')
    for idx,(group,left,right) in enumerate(peer_pairs):
        row=3+len(own_pairs)+idx
        for col,ss in [(0,left),(1,right)] if right else [(0,left)]:
            ax=fig.add_subplot(gs[row,col] if right else gs[row,:])
            ax.set_facecolor('#fcf9ff')
            for kk in ids:
                if lot_meta.loc[kk,'MINUTES']<=a:continue
                for s in ss:
                    start=len(ax.lines)
                    line(ax,kk,s,start=a,end=min(b,lot_meta.loc[kk,'MINUTES']),raw_values=True)
                    for artist in ax.lines[start:]:
                        artist.set_linewidth(2.4 if kk==k else 1.3)
                        artist.set_alpha(1 if kk==k else .8)
                        if s=='CP-Z4M':
                            artist.set_ydata(np.asarray(artist.get_ydata())*sensor_scale(s))
                    compared.add(s)
            if ss[0]=='CP-Z4M':
                ax.relim();ax.autoscale_view();ax.ticklabel_format(axis='y',style='plain',useOffset=False)
            d=comparison[k,ss[0]].reindex(range(int(np.floor(a)),int(np.ceil(b))))
            for aa,bb in runs(d.DELTA.isna()):
                ax.axvspan(max(a,aa),min(b,bb),color='#e9edf0',alpha=.55,lw=0)
            ax.set_xlim(a,b);clock_top(ax,r.START,label=f'대상 {k} 실제 시각[시:분]')
            style(ax);ax.set_ylabel(shown_unit(ss[0]));ax.tick_params(labelsize=12)
            ax.set_title(f"같은 강종 LOT · {group} [{' · '.join(ss)}] · 경과시간 정렬",loc='left',fontsize=15,pad=42)
            mark_peer_condition(fig,ax,k,ss[0],a,b)
            axis_modes.append('ELAPSED')
    assert sensors<=observed, '후보 센서가 자체 신호에서 빠짐'
    assert sensors<=compared, '후보 센서가 강종 대조에서 빠짐'
    header(fig,f'{k} · 공정별 신호와 강종별 LOT 대조',f"{r.GRADE} · {when(k,a,b)} · 1초 원본 · 전체 위치 → 공정별 비교 → 같은 강종 LOT 비교",[],ncol=5)
    fig._suptitle.set_y(1-.35/height)
    compact_header(fig,1-1.15/height,1-1.75/height,title_size=24,sub_size=13,legend_size=12)
    axis_tag(fig,'TIME AXIS · 대상: 실제 시각 / 강종 대조: 경과시간')
    fig.texts[-1].set_y(1-.35/height)
    def section_header(slot,text,background,accent,handles):
        section=fig.add_subplot(gs[slot,:]);section.axis('off');p0=section.get_position()
        band_height=.48/height;section_left=.035
        fig.add_artist(Rectangle((section_left,p0.y1-band_height),p0.x1-section_left,band_height,
            transform=fig.transFigure,facecolor=background,edgecolor=accent,lw=1.1))
        fig.text(section_left+.01,p0.y1-band_height/2,text,ha='left',va='center',
            fontsize=22,fontweight='bold',color=accent)
        fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(section_left,p0.y1-.6/height),
            ncol=5 if len(handles)>6 else len(handles),frameon=False,fontsize=14,
            handlelength=2.5,columnspacing=1.8,labelspacing=.55)
    section_header(1,f'공정별 비교 · 대상 LOT {k} | 선 색 = 공정 구역 · 실제 측정 시각',
        '#e3f0fc','#226aa9',zone_handles)
    section_header(2+len(own_pairs),f'강종별 비교 · {r.GRADE} LOT 대조 | 선 색 = LOT 번호 · 경과시간',
        '#eee6f7','#7650a4',peer_handles)
    finish(fig,name)
    note('**한 그림에서 읽는 순서:** 전체 파형의 상자로 위치를 찾고, 파란 공정별 비교에서 대상 LOT의 구역별 온도·출력·CP를 같은 실제 시각에서 확인한다. '
        '보라 강종별 비교에서는 해당 센서를 LOT 색으로 구분하고 대상 LOT을 굵게 그렸다. 범례를 두 부분 바로 위에 따로 두었다. '
        '다른 LOT의 실제 날짜는 서로 다르며, 경과시간을 맞춘 비교다. '
        '**패널 옆 대조 표시:** 대상 LOT을 제외한 같은 강종의 안정 참고·분당 유효 기록 90% 이상 자료를 센다. '
        '2+ LOT은 2개 이상, 1 LOT은 한 개와의 참고 비교다. %는 확대 시간 중 각 조건이 성립하는 비율이다. '
        '불가·회색은 대상 또는 대조가 비교 조건을 충족하지 않는 시간이며 기록 누락만을 뜻하지 않는다.')
    for rr in evidence.itertuples():
        g=raw[k].loc[raw[k].ELAPSED.ge(rr.A)&raw[k].ELAPSED.lt(rr.B)]
        note(f"**{rr.REVIEW_ID} · {rr.SENSOR}:** {rr.DETAIL}. 실제 {rr.START:%m/%d %H:%M:%S}–{rr.END:%m/%d %H:%M:%S}, "
             f"원본 유효 기록 {g[rr.SENSOR].count()}개, 범위 {g[rr.SENSOR].min():.4g}~{g[rr.SENSOR].max():.4g}.")
    window_coverage.setdefault(k,set()).update(win['IDS'])
    window_audit.append({'LOT':k,'WINDOW':win['WINDOW_ID'],'FILE':name,'MODE':'CLOCK+PEERS','REVIEW_IDS':win['IDS'],'A':a,'B':b})
    combined_window_audit.append({'LOT':k,'WINDOW':win['WINDOW_ID'],'FILE':name,'REVIEW_IDS':win['IDS'],
        'OBSERVED':sorted(observed),'COMPARED':sorted(compared),'AXIS_MODES':axis_modes,
        'OWN_PAIRS':len(own_pairs),'PEER_PAIRS':len(peer_pairs),'PEER_LOTS':ids,
        'SECTION_HEADERS':['공정별 비교','강종별 비교'],'CPM_DISPLAY_SCALE':sensor_scale('CP-Z4M')})


In [ ]:
def lot_history(k):
    r=lot_meta.loc[k];tasks=linked_work(k,pad_hours=72)
    rows=[]
    for t in tasks.itertuples():
        overlap=max(0,(min(r.END,t.END_DT)-max(r.START,t.STRT_DT)).total_seconds()/60)
        detection_in=r.START<=t.DETC_DT<=r.END
        rows.append({'LOT':k,'WO':t.WO_NO,'유형':t.MNT_TYPE,'관계':'작업 교집합' if overlap else '관측 중 감지' if detection_in else '관측 밖 인근 이력',
                     '감지':t.DETC_DT,'착수':t.STRT_DT,'완료':t.END_DT,'관측·작업 교집합[분]':overlap,
                     '관측 종료 → 착수[분]':(t.STRT_DT-r.END).total_seconds()/60,'작업 내용':remark_text(t),
                     '확인된 교체 부품':'; '.join(parts_for(t.WO_NO))})
    if rows:table(pd.DataFrame(rows).rename(columns={'유형':'유형'}))
    note('**이력 관계:** 위 표는 관측 ±72시간에 있는 CAL01 이력이다. 관측 밖 가까운 작업과 실제 교집합을 구분하며, 시간상 가까움만으로 신호의 원인·결과를 연결하지 않는다. 확인된 부품만 같은 WO로 표시한다.')

def cp_event_flow(k,name):
    r=lot_meta.loc[k];tasks=linked_work(k,pad_hours=12)
    if not len(tasks) or k not in set(cp_candidates.LOT):return
    c=cp_candidates.set_index('LOT').loc[k]
    lo=r.START-pd.Timedelta(minutes=10);hi=max(r.END,tasks.END_DT.max())+pd.Timedelta(minutes=30)
    fig,(ax,events)=plt.subplots(2,1,figsize=(28,13),sharex=True,gridspec_kw={'height_ratios':[2.2,1]})
    fig.subplots_adjust(left=.09,right=.975,top=.78,bottom=.10,hspace=.48)
    line(ax,k,'CP-Z4',clock=True,envelope=True)
    ax.axhline(c.BASE,color='#455a64',ls='--',lw=1)
    for a in [ax,events]:
        a.axvline(c.PEAK_TIME,color=COL['FOCUS'],lw=1.5);a.axvline(r.END,color='#455a64',ls=':',lw=1.3)
    for row,t in enumerate(tasks.itertuples()):
        events.plot([t.STRT_DT,t.END_DT],[row,row],color=COL[t.MNT_TYPE],lw=10,solid_capstyle='butt')
        events.scatter(t.DETC_DT,row,marker='v',s=80,color=COL['DETECT'])
        if parts_for(t.WO_NO):events.scatter(t.END_DT,row,marker='D',s=65,color=COL['PART'])
    events.set_yticks(range(len(tasks)),[f'{t.MNT_TYPE} · {t.WO_NO}\n{remark_text(t)}' for t in tasks.itertuples()])
    events.set_ylim(len(tasks)-.5,-.5);ax.set_xlim(lo,hi)
    for a in [ax,events]:clock_axis(a);a.tick_params(labelsize=12)
    ax.set_title('(a) CP 전체 관측과 극값 위치',loc='left',fontsize=16,pad=12)
    events.set_title('(b) 같은 실제 시각의 정비 이력',loc='left',fontsize=16,pad=12)
    header(fig,f'{k} CP 변화·정비 시각 대조',f'{r.GRADE} · 점선: 자체 참고 중앙값 · 실제 날짜·시각에 따른 기록 대조',
           [Line2D([0],[0],color=COL['FOCUS'],lw=1.5,label='CP 극값 시각'),Line2D([0],[0],color='#455a64',ls=':',label='제공 관측 끝'),
            *[Patch(facecolor=COL[x],label=x+' 작업') for x in tasks.MNT_TYPE.unique()],
            Line2D([0],[0],marker='v',color=COL['DETECT'],ls='None',label='감지 기록'),Line2D([0],[0],marker='D',color=COL['PART'],ls='None',label='교체 이력 · WO 완료 위치')],ncol=6)
    compact_header(fig,.935,.905,title_size=24,sub_size=13,legend_size=12);axis_tag(fig,TAG_REAL);finish(fig,name)
    for t in tasks.itertuples():
        if c.PEAK_TIME > t.END_DT:
            relation=f'작업 완료 {(c.PEAK_TIME-t.END_DT).total_seconds()/60:.2f}분 뒤 CP 극값'
        elif c.PEAK_TIME < t.STRT_DT:
            relation=f'CP 극값 {(t.STRT_DT-c.PEAK_TIME).total_seconds()/60:.2f}분 뒤 작업 착수'
        else:
            relation='실제 작업 중 CP 극값'
        note(f'**기록 시각 대조:** {t.WO_NO}: 감지 {t.DETC_DT:%m/%d %H:%M}, 작업 {t.STRT_DT:%m/%d %H:%M}–{t.END_DT:%m/%d %H:%M} [{remark_text(t)}]. '
             f'CP 극값 {c.PEAK_TIME:%m/%d %H:%M:%S} · {relation}. 제공 관측 끝 {r.END:%m/%d %H:%M:%S}.')
        if parts_for(t.WO_NO):note('**같은 WO 교체:** '+'; '.join(parts_for(t.WO_NO))+'. WO 완료 위치는 실제 교체 순간이 아니다.')
    if k==240117:
        start=r.START+pd.Timedelta(minutes=float(c.START_MIN));end=r.START+pd.Timedelta(minutes=float(c.END_MIN))
        before=raw[k].loc[raw[k].MEAS_DT.ge(start-pd.Timedelta(minutes=10))&raw[k].MEAS_DT.lt(start),'CP-Z4'].median()
        after=raw[k].loc[raw[k].MEAS_DT.ge(end)&raw[k].MEAS_DT.lt(end+pd.Timedelta(minutes=10)),'CP-Z4'].median()
        note(f'**관측 양상:** CP-Z4는 하강 직전 10분 중앙값 {before:.3f}, 극값 {c.PEAK_VALUE:.3f}, 직후 10분 중앙값 {after:.3f}다. '
             '짧은 하강 뒤 평소 수준으로 돌아왔다. 마모량 초과는 정비 기록의 내용이며, 이 CP 하강의 원인이나 CP 복귀의 정비 효과를 뜻하지 않는다. '
             '마모 부품·마모량과 CP 센서/계산/분위기 제어 기록을 함께 확인해야 한다.')
    note('**확인 범위:** 시간 선후관계와 작업 중 여부는 확인할 수 있다. CP 변화가 작업의 원인이나 결과인지는 이 순서만으로 판단할 수 없다. 관측이 끝난 뒤의 센서 값은 그리지 않는다.')


In [ ]:
PAGE_SIGNALS={"세정·건조":[['CLN-IN'],['TC-Z1','OP-Z1'],['TC-Z2','OP-Z2']],
"가열·균열·CP":[['TC-Z3','OP-Z3'],['TC-Z4','OP-Z4'],['CP-Z4','CP-Z4M']],
"가열·과시효":[['TC-Z5','OP-Z5'],['TC-Z6','OP-Z6'],['TC-Z7','OP-Z7'],['TC-Z8','OP-Z8']],
"Z9·냉각":[['TC-Z9'],['TC-OUT1','TC-OUT2']]}
def grade_compare(grade,page,name):
    ids=lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int).tolist();rows=PAGE_SIGNALS[page]
    fig,axes=plt.subplots(len(rows),2,figsize=(28,4+4.3*len(rows)),squeeze=False)
    fig.subplots_adjust(left=.085,right=.975,top=.84,bottom=.055,wspace=.22,hspace=.75)
    for row,ss in enumerate(rows):
        for col in range(2):
            ax=axes[row,col]
            if col>=len(ss):ax.set_axis_off();continue
            s=ss[col]
            for k in ids:
                p=profile[k];scale=sensor_scale(s)
                ax.plot(p.index,p[(s,'median')]*scale,color=LOT_COLOR[k],lw=1.7)
                ax.fill_between(p.index,p[(s,'min')]*scale,p[(s,'max')]*scale,color=LOT_COLOR[k],alpha=.08,lw=0)
                for e in episodes.loc[episodes.LOT.eq(k)&episodes.SENSOR.eq(s)].itertuples():ax.axvspan(e.START_MIN,e.END_MIN,color=LOT_COLOR[k],alpha=.08,lw=0)
            ax.set_title(f"{PROCESS[s]} [{s}]",loc='left',fontsize=15,pad=12)
            ax.set_ylabel(shown_unit(s));style(ax);ax.set_xlim(0,lots.loc[lots.GRADE.eq(grade),'MINUTES'].max());ax.tick_params(labelsize=12)
    header(fig,f"{grade} · {page} 같은 강종 비교","선: 1분 중앙값 · 음영: 원본 최소–최대 · 강조 시간창: 같은 강종 대비 지속 차이",lot_handles(ids)+[QUALITY_HANDLE],ncol=5)
    compact_header(fig,.945,.915,title_size=24,sub_size=13,legend_size=12);axis_tag(fig,TAG_ELAPSED);finish(fig,name)

def grade_peer_note(grade):
    """별도의 차이·대조 범위 그림을 만들지 않고 전체 비교 아래 한 줄만 안내한다."""
    ids = lots.loc[lots.GRADE.eq(grade), 'LOT_NO'].astype(int).tolist()
    lengths = sorted((float(lot_meta.loc[k, 'MINUTES']) for k in ids), reverse=True)
    if len(ids) == 2:
        text = f'같은 강종 2 LOT → 다른 1 LOT과의 참고 비교. 공통 관측창은 경과 {lengths[1]:.1f}분까지다.'
    else:
        text = (f'같은 강종 {len(ids)} LOT → 대상 제외 최대 {len(ids)-1} LOT. '
                f'다른 2 LOT 이상을 확보할 수 있는 관측창은 최대 경과 {lengths[2]:.1f}분까지다.')
    note('**대조 조건:** ' + text + ' 센서별 안정 참고·기록량 조건에 따라 일부 시간은 제외되며, 해당 확대 패널 옆에서 실제 대조 수를 확인한다.')


In [ ]:
ZONES = [3, 4, 5, 6]
ZONE_COLOR = {3: "#0072b2", 4: "#e69f00", 5: "#009e73", 6: "#cc79a7"}   # 구역마다 다른 색 · LOT은 제목 색으로 구분
GRADE_COLOR = {"SGCC": "#455a64", "SPFH590": "#d55e00", "SPCE": "#0072b2", "SPCEN": "#e69f00", "SPHC": "#009e73",
               "DP590": "#a23b57", "SPCD": "#7b2cbf", "SPCC": "#8c6d31"}

def remark_text(task):
    return str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재"

def part_text(wo):
    d = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    return ", ".join(f"{p.PART_NM} ×{int(p.QTY)}({p.CHG_RSN})" for p in d.itertuples())

def work_label(t):
    """작업 표시: REMARK · 작업 길이 · FAIL_MODE 코드 · 교체 부품."""
    text = f"{remark_text(t)} · {(t.END_DT - t.STRT_DT).total_seconds() / 60:.0f}분"
    if pd.notna(t.FAIL_MODE):
        text += f" · FAIL_MODE {t.FAIL_MODE}"
    if part_text(t.WO_NO):
        text += f" · 교체: {part_text(t.WO_NO)}"
    return text

def grade_periods(lo, hi):
    """lo~hi 사이 강종 구간: 각 강종변경부터 다음 변경까지."""
    ev = grade_evt.loc[grade_evt.EVT_DT.lt(hi)]
    ev = pd.concat([ev.loc[ev.EVT_DT.le(lo)].tail(1), ev.loc[ev.EVT_DT.gt(lo)]])
    starts = [max(t, lo) for t in ev.EVT_DT]
    return list(zip(starts, starts[1:] + [hi], ev.AFT_VAL))

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a*60)), min(r.END, r.START + pd.Timedelta(seconds=int(b*60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def zone_panel(ax, lot, a, b, kind, title):
    w = raw_window(lot, a, b)
    for z in ZONES:
        ax.plot(w.index, w[f"{kind}-Z{z}"], color=ZONE_COLOR[z], lw=1.4, label=f"{kind}-Z{z}")
    # 범례는 그림 오른쪽 위(제목과 같은 높이)에 둔다.
    leg = ax.legend(loc="lower right", bbox_to_anchor=(1, 1), borderaxespad=3, ncol=4, frameon=False,
                    fontsize=13, handlelength=1.8, columnspacing=1.2)
    for line_ in leg.get_lines():
        line_.set_linewidth(3.5)
    ax.set_xlim(w.index[0], w.index[-1])
    ax.set_title(title, loc="left", fontsize=15, pad=42, color=LOT_COLOR[lot])
    ax.set_ylabel("온도[℃]" if kind == "TC" else "제어출력[%]", labelpad=8)
    style(ax, f"실제 측정 시각[{w.index[0]:%m/%d} 시:분]")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(axis="both", labelsize=12.5)
    elapsed_top(ax, lot_meta.loc[lot, "START"])
    return w

def case_figure(plan, before, after, after_window, title, extra=None, more_handles=()):
    rb, ra = lot_meta.loc[before], lot_meta.loc[after]
    tail = float(state_summary.set_index("LOT").loc[before, "TAIL_START_MIN"])
    b_win = (max(0, tail - 10), rb.MINUTES)
    lo, hi = rb.START - pd.Timedelta(hours=6), ra.END + pd.Timedelta(hours=6)
    span = hi - lo
    ev = grade_evt.loc[grade_evt.EVT_DT.between(lo, hi)]
    tasks = cal_work.loc[((cal_work.STRT_DT < hi) & (cal_work.END_DT > lo)) | cal_work.DETC_DT.between(lo, hi)].sort_values("STRT_DT")

    fig = plt.figure(figsize=(30, 31))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.9, 1, 1], left=.1, right=.975, top=.87, bottom=.04, hspace=.52, wspace=.14)
    axa = fig.add_subplot(gs[0, :])
    rows = ["강종 구간", "온도 관측"] + [f"{t.MNT_TYPE}\n{t.WO_NO}" for t in tasks.itertuples()]
    axa.axvspan(plan.STRT_DT, plan.END_EXCL, color=COL["PLAN"], alpha=.45, lw=0)
    # 강종 구간: 변경 시각부터 다음 변경까지 강종 색 띠, 띠 위 상자는 변경 시각
    for a, b_, g in grade_periods(lo, hi):
        axa.plot([a, b_], [0, 0], color=GRADE_COLOR.get(g, "#888888"), lw=30, solid_capstyle="butt", alpha=.9)
        if b_ - a > span * .045:
            axa.text(a + (b_ - a) / 2, 0, g, ha="center", va="center", fontsize=15, color="white", fontweight="bold")
    for i, e in enumerate(ev.itertuples()):
        level = -1.15 if i % 2 else -.6
        axa.plot([e.EVT_DT, e.EVT_DT], [-.3, level], color="#17344d", lw=1)
        axa.text(e.EVT_DT, level, f"{e.EVT_DT:%m/%d %H:%M} {e.BEF_VAL}→{e.AFT_VAL}", ha="center", va="bottom", fontsize=13,
                 color="#17344d", bbox=dict(boxstyle="round,pad=.25", fc="white", ec="#17344d", lw=.7))
    for lot in [before, after]:
        r = lot_meta.loc[lot]
        axa.plot([r.START, r.END], [1, 1], color=LOT_COLOR[lot], lw=11, solid_capstyle="butt")
    for i, t in enumerate(tasks.itertuples(), start=2):
        axa.plot([t.STRT_DT, t.END_DT], [i, i], color=COL[t.MNT_TYPE], lw=9, solid_capstyle="butt")
        axa.plot(t.DETC_DT, i, "v", color=COL["DETECT"], ms=11)
        if part_text(t.WO_NO):
            axa.plot(t.END_DT, i, "D", color=COL["PART"], ms=11)
        right = t.END_DT < lo + span * .55
        x = t.END_DT + span * .008 if right else t.DETC_DT - span * .008
        axa.text(x, i, work_label(t), ha="left" if right else "right", va="center", fontsize=13,
                 color=COL["PART"] if part_text(t.WO_NO) else COL[t.MNT_TYPE],
                 fontweight="bold" if part_text(t.WO_NO) else "normal",
                 bbox=dict(boxstyle="round,pad=.25", fc="white", ec=COL["PART"] if part_text(t.WO_NO) else COL[t.MNT_TYPE], lw=.7))
    axa.set_yticks(range(len(rows)), rows)
    for tick, color in zip(axa.get_yticklabels(), ["#17344d", COL["LOT"]] + [COL[t] for t in tasks.MNT_TYPE]):
        tick.set_color(color)
    axa.set_ylim(len(rows) - .4, -1.75)
    axa.set_xlim(lo, hi)
    style(axa, "2024년 실제 기록 시각[월/일]")
    axa.xaxis.set_major_locator(mdates.DayLocator())
    axa.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
    axa.tick_params(axis="both", labelsize=13.5)
    axa.set_title(f"(a) 전체 사건 흐름 · 계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}({plan.SCOPE}) · CAL01 실제 작업 {len(tasks)} WO",
                  loc="left", fontsize=16, pad=14)
    for key, lot, (a, b_) in [("b", before, b_win), ("c", after, after_window)]:
        r = lot_meta.loc[lot]
        x0, x1 = r.START + pd.Timedelta(minutes=a), r.START + pd.Timedelta(minutes=b_)
        pad = span * .004
        axa.add_patch(Rectangle((mdates.date2num(x0 - pad), .55), mdates.date2num(x1 + pad) - mdates.date2num(x0 - pad), .9,
                                fill=False, ec=COL["FOCUS"], lw=2.2, zorder=6))
        axa.text(x0 + (x1 - x0) / 2, 1.62, f"({key})", ha="center", va="top", fontsize=15, fontweight="bold", color=COL["FOCUS"])
    bt, bo = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    ct, co = fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])
    zone_panel(bt, before, *b_win, "TC", f"(b) 수리 전 {before} · {rb.GRADE} · 종료부 · 가열·균열 [TC-Z3~Z6]")
    zone_panel(bo, before, *b_win, "OP", f"(b) 수리 전 {before} · {rb.GRADE} · 종료부 · 가열·균열 [OP-Z3~Z6]")
    label = "전체 관측" if after_window[1] >= ra.MINUTES else "초기"
    zone_panel(ct, after, *after_window, "TC", f"(c) 수리 후 {after} · {ra.GRADE} · {label} · 가열·균열 [TC-Z3~Z6]")
    zone_panel(co, after, *after_window, "OP", f"(c) 수리 후 {after} · {ra.GRADE} · {label} · 가열·균열 [OP-Z3~Z6]")
    for ax in [bt, bo]:
        ax.axvspan(rb.START + pd.Timedelta(minutes=tail), rb.END, color=COL["FOCUS"], alpha=.08, lw=0)
    if extra is not None:
        extra(ct, co)
    handles = (lot_handles([before, after]) +
               [Patch(facecolor=COL["PLAN"], alpha=.6, label="CR1 계획수리 날짜"),
                *[Line2D([0], [0], color=COL[k], lw=6, label=f"{k} 실제 작업") for k in tasks.MNT_TYPE.drop_duplicates()],
                Line2D([0], [0], color=COL["DETECT"], marker="v", ls="None", ms=10, label="DETC_DT 감지 · 방법 미제공"),
                Line2D([0], [0], color=COL["PART"], marker="D", ls="None", ms=10, label="부품교체 · WO 완료 기준"),
                Patch(facecolor="white", edgecolor=COL["FOCUS"], lw=2, label="(a)의 확대 위치 → (b)·(c)"), *more_handles])
    header(fig, title, "(a) 실제 날짜의 강종 구간·관측·작업 → (b) 수리 전 종료부 → (c) 수리 후 관측; (b)·(c) 선 색 = 구역 Z3~Z6(각 그림 오른쪽 위 범례), 제목 색 = LOT, 위쪽 축 = 같은 LOT 경과 분",
           handles, ncol=6)
    compact_header(fig, .956, .936)
    axis_tag(fig, TAG_REAL)
    return fig, tasks, ev, b_win

def heat_delta(lot, kind):
    g, r = raw[lot], lot_meta.loc[lot]
    cols = HEAT + [f"OP-Z{z}" for z in ZONES]
    if kind == "종료":
        d = g.loc[g.ELAPSED.ge(r.MINUTES - 5), cols].median() - g.loc[g.ELAPSED.between(r.MINUTES*.45, r.MINUTES*.6), cols].median()
    else:
        d = g.loc[g.ELAPSED.between(80, 100), cols].median() - g.loc[g.ELAPSED.lt(5), cols].median()
    return d.loc[HEAT], d.loc[[f"OP-Z{z}" for z in ZONES]]

def detect_data(tasks):
    """각 감지 시각 ±10분의 온도 기록 행 수."""
    return {t.WO_NO: int(temp.MEAS_DT.between(t.DETC_DT - pd.Timedelta(minutes=10), t.DETC_DT + pd.Timedelta(minutes=10)).sum())
            for t in tasks.itertuples()}

def work_table(tasks):
    near = detect_data(tasks)
    return pd.DataFrame([{"WO": t.WO_NO, "종류": t.MNT_TYPE, "DETC_DT": t.DETC_DT, "감지 ±10분 온도[행]": near[t.WO_NO],
                          "작업 시작": t.STRT_DT, "작업 완료": t.END_DT, "작업 기록 길이[분]": (t.END_DT - t.STRT_DT).total_seconds() / 60,
                          "DOWN_MIN 기록[분]": t.DOWN_MIN, "FAIL_MODE 코드": t.FAIL_MODE, "REMARK 원문": t.REMARK,
                          "교체 부품·수량·사유": part_text(t.WO_NO) or "-"} for t in tasks.itertuples()])

In [ ]:
# CAL01 정비 경계와 실제 온도 기록의 교집합을 먼저 확인한다.
EVENT_PAD_MIN = 25
NEAREST_OBSERVATION_MIN = 30
_sensor_times = temp.MEAS_DT.sort_values().reset_index(drop=True)
EVENT_SPECS = [("DETC_DT", "감지", ":"), ("STRT_DT", "착수", "--"), ("END_DT", "완료", "-.")]
ZOOM_GROUPS = [
    ("가열·균열 TC-Z3~Z6", HEAT),
    ("가열·균열 OP-Z3~Z6", [f"OP-Z{z}" for z in ZONES]),
    ("CP-Z4", ["CP-Z4"]), ("CP-Z4M", ["CP-Z4M"]),
]


def measured_rows(start, end):
    return int(_sensor_times.searchsorted(end, side="right") - _sensor_times.searchsorted(start, side="left"))


def zoom_parts(wo):
    records = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    return [f"{r.PART_NM} [{r.PART_CD}] ×{r.QTY:g} ({r.CHG_RSN})" for r in records.itertuples()]


maintenance_zoom_audit = []
for task in cal_work.itertuples():
    row = {"WO": task.WO_NO, "유형": task.MNT_TYPE, "작업 내용": remark_text(task)}
    for field, label, _ in EVENT_SPECS:
        at = getattr(task, field)
        row[label + " 시각"] = at
        row[label + " 시각 측정 기록"] = bool(measured_rows(at, at))
        row[label + " ±25분 기록[행]"] = measured_rows(at-pd.Timedelta(minutes=EVENT_PAD_MIN), at+pd.Timedelta(minutes=EVENT_PAD_MIN))
    row["작업 중 관측 겹침[분]"] = sum(
        max(0, (min(task.END_DT, lot.END)-max(task.STRT_DT, lot.START)).total_seconds()/60)
        for lot in lots.itertuples())
    row["기록된 교체 부품"] = "; ".join(zoom_parts(task.WO_NO))
    maintenance_zoom_audit.append(row)
maintenance_zoom_audit = pd.DataFrame(maintenance_zoom_audit)
observed_event_wos = maintenance_zoom_audit.loc[
    maintenance_zoom_audit[[name+" ±25분 기록[행]" for _,name,_ in EVENT_SPECS]].gt(0).any(axis=1), "WO"]


def merge_event_windows(events, pad_min):
    """맞닿거나 겹치는 검토창은 한 번만 그린다. 떨어진 시각은 별도 창을 유지한다."""
    spans = sorted((r["time"]-pd.Timedelta(minutes=pad_min), r["time"]+pd.Timedelta(minutes=pad_min)) for r in events)
    merged = []
    for left,right in spans:
        if merged and left <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(right,merged[-1][1]))
        else:
            merged.append((left,right))
    return merged


def measurement_gap_spans(grid, present):
    """음영은 양쪽 관측점까지 연결하고, 공백 수는 실제 빠진 예상 시각으로 센다."""
    missing = np.flatnonzero(~present)
    if not len(missing):
        return []
    groups = np.split(missing, np.flatnonzero(np.diff(missing)>1)+1)
    return [(grid[max(0,int(a[0])-1)], grid[min(len(grid)-1,int(a[-1])+1)], len(a)) for a in groups]


maintenance_zoom_windows = []
maintenance_zoom_checks = []


def maintenance_zoom(wo_numbers, name, title, pad_min=EVENT_PAD_MIN,
                     nearest_minutes=NEAREST_OBSERVATION_MIN, groups=ZOOM_GROUPS):
    """정비 경계의 1초 원본. 경계에 기록이 없으면 날짜상 직전 제공 관측을 별도로 표시한다."""
    tasks = cal_work.loc[cal_work.WO_NO.isin(wo_numbers)].sort_values("STRT_DT")
    assert set(tasks.WO_NO)==set(wo_numbers), "CAL01에 없는 WO"
    events = [{"WO":t.WO_NO,"type":t.MNT_TYPE,"label":label,"style":style_,"time":getattr(t,field)}
              for t in tasks.itertuples() for field,label,style_ in EVENT_SPECS]
    requested = merge_event_windows(events,pad_min)
    windows = [(a,b,"정비 경계") for a,b in requested if measured_rows(a,b)>0]
    fallback = not windows
    if fallback:
        first_event = min(e["time"] for e in events)
        prior = lots.loc[lots.END.lt(first_event)].sort_values("END").tail(1)
        assert len(prior), "직전 제공 LOT도 없어 확대할 센서 자료가 없음"
        nearest = prior.iloc[0]
        gap = (first_event-nearest.END).total_seconds()/3600
        assert gap <= 72, "직전 제공 관측도 72시간 밖: 작업 시각만 이력 표에서 확인"
        windows = [(max(nearest.START,nearest.END-pd.Timedelta(minutes=nearest_minutes)),nearest.END,"직전 제공 관측")]

    # 각 창은 실제 날짜·시각을 보존한다. 다른 LOT의 선을 서로 이어 붙이지 않는다.
    window_lots = []
    for left,right,kind in windows:
        observed = lots.loc[lots.START.le(right)&lots.END.ge(left)].sort_values("START")
        assert len(observed), "관측이 없는 창은 원본 확대에 넣지 않음"
        window_lots.append(observed)
    shown_lots = pd.concat(window_lots).drop_duplicates("LOT_NO").sort_values("START")
    ncols = len(windows)
    width = max(24,3+9*ncols)
    fig = plt.figure(figsize=(width,6+3.5*len(groups)))
    grid_spec = fig.add_gridspec(len(groups)+1,ncols,
        height_ratios=[1.8]+[1]*len(groups),left=.14,right=.975,top=.85,bottom=.05,
        wspace=.25,hspace=.6)
    timeline = fig.add_subplot(grid_spec[0,:])
    timeline_labels = []
    lane = 0
    for lot in shown_lots.itertuples():
        timeline.plot([lot.START,lot.END],[lane,lane],color=GRADE_COLOR[lot.GRADE],lw=9,solid_capstyle="butt")
        timeline_labels.append(f"온도 {lot.LOT_NO} · {lot.GRADE}")
        lane += 1
    task_lanes = {}
    for task in tasks.itertuples():
        task_lanes[task.WO_NO] = lane
        timeline.plot([task.STRT_DT,task.END_DT],[lane,lane],color=COL[task.MNT_TYPE],lw=8,solid_capstyle="butt")
        timeline.scatter(task.DETC_DT,lane,marker="v",s=75,color=COL["DETECT"],zorder=5)
        timeline.scatter([task.STRT_DT,task.END_DT],[lane,lane],marker="|",s=170,color=COL[task.MNT_TYPE],zorder=5)
        timeline_labels.append(f"{task.MNT_TYPE} · {task.WO_NO}\n{remark_text(task)}")
        lane += 1
        for part_record in zoom_parts(task.WO_NO):
            timeline.scatter(task.END_DT,lane,marker="D",s=65,color=COL["PART"],zorder=5)
            timeline_labels.append("교체 기록\n" + part_record)
            lane += 1
    # 본창 이전 감지는 시각을 옮기지 않고 아래 이력 표에 보존한다.
    # 직전 관측 확대에서는 해당 30분과 BM의 사건 순서에 집중한다.
    lo = min(min(a for a,_,_ in windows),tasks.STRT_DT.min())
    if not fallback:
        lo = min(lo,shown_lots.START.min())
    hi = max(max(e["time"] for e in events),max(b for _,b,_ in windows),shown_lots.END.max())
    margin = max((hi-lo)*.025,pd.Timedelta(minutes=5))
    timeline.set_xlim(lo-margin,hi+margin)
    timeline.set_ylim(lane-.45,-.85)
    timeline.set_yticks(range(lane),timeline_labels)
    timeline.tick_params(axis="both",labelsize=12.5)
    style(timeline,"실제 기록 날짜·시각 · 사각형 (a)~: 아래 원본 확대 범위")
    timeline.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=7))
    timeline.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    timeline.set_title("관측·감지·실제 작업·부품교체 이력",loc="left",fontsize=16,pad=15)

    for column,((left,right,kind),observed) in enumerate(zip(windows,window_lots)):
        letter = chr(97+column)
        timeline.axvspan(left,right,color=COL["FOCUS"],alpha=.09,lw=0)
        timeline.text(left+(right-left)/2,-.55,f"({letter})",ha="center",va="center",
                      fontsize=15,fontweight="bold",color=COL["FOCUS"])
        times = pd.date_range(left,right,freq="s")
        subset = temp.loc[temp.MEAS_DT.between(left,right)].set_index("MEAS_DT")
        assert subset.index.is_unique
        measured = subset.reindex(times)
        gaps = measurement_gap_spans(times,times.isin(subset.index))
        primary_lot = int(observed.iloc[0].LOT_NO)
        in_window = [e for e in events if left<=e["time"]<=right]
        axes = []
        for row,(group_name,sensors) in enumerate(groups):
            ax = fig.add_subplot(grid_spec[row+1,column],sharex=axes[0] if axes else None)
            axes.append(ax)
            for lot in observed.itertuples():
                valid_lot = measured.LOT_NO.eq(lot.LOT_NO)
                for sensor in sensors:
                    color = ZONE_COLOR[int(sensor.rsplit("Z",1)[1])] if sensor.startswith(("TC-Z","OP-Z")) and int(sensor.rsplit("Z",1)[1]) in ZONE_COLOR else GRADE_COLOR[lot.GRADE]
                    y = measured[sensor].where(valid_lot)
                    ax.plot(times,y,color=color,lw=.95 if len(sensors)>1 else 1.3)
            for start,end,missing_seconds in gaps:
                ax.axvspan(start,end,color="#d8dfe5",alpha=.8,lw=0,zorder=0)
                if row==0 and missing_seconds>=300:
                    ax.text(start+(end-start)/2,.12,"측정 기록 없음",transform=ax.get_xaxis_transform(),
                            ha="center",va="center",fontsize=11,color="#526d80")
            for event in in_window:
                color = COL["DETECT"] if event["label"]=="감지" else COL[event["type"]]
                ax.axvline(event["time"],color=color,ls=event["style"],lw=1.6,zorder=4)
            xlabel = f"{left:%m/%d} 실제 측정 시각[시:분]" if row==len(groups)-1 else ""
            style(ax,xlabel)
            ax.set_xlim(left,right)
            ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
            ax.tick_params(axis="both",labelsize=12,labelbottom=row==len(groups)-1)
            ax.yaxis.set_major_locator(MaxNLocator(4))
            ax.set_ylabel("온도[℃]" if sensors[0] in TC else "제어출력[%]" if sensors[0] in OP else unit(sensors[0]),fontsize=13,labelpad=8)
            if sensors==["CP-Z4M"]:
                ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
            heading = f"({letter}) {primary_lot} · {kind} · {group_name}" if row==0 else group_name
            ax.set_title(heading,loc="left",pad=30,fontsize=14.5,color=GRADE_COLOR[lot_meta.loc[primary_lot,"GRADE"]] if row==0 else "#17344d")
        count = measured_rows(left,right)
        maintenance_zoom_windows.append({
            "그림":name,"창":f"({letter})","종류":kind,"LOT":", ".join(str(int(k)) for k in observed.LOT_NO),
            "확대 시작":left,"확대 종료":right,"측정 기록[행]":count,
            "창 안의 정비 경계":" / ".join(f"{e['type']} {e['WO'].rsplit('-',1)[-1]} {e['label']} {e['time']:%m/%d %H:%M}" for e in in_window)})
        assert all(ax.get_xlim()==axes[0].get_xlim() for ax in axes)

    handles = [Line2D([0],[0],color=ZONE_COLOR[z],lw=2,label=f"Z{z} 온도·출력") for z in ZONES]
    handles += [Line2D([0],[0],color=COL["DETECT"],ls=":",lw=1.7,label="DETC_DT 감지 기록"),
                Line2D([0],[0],color="#455a64",ls="--",lw=1.7,label="작업 착수"),
                Line2D([0],[0],color="#455a64",ls="-.",lw=1.7,label="작업 완료"),
                Patch(facecolor="#d8dfe5",label="측정 기록 공백"),
                Patch(facecolor=COL["FOCUS"],alpha=.25,label="원본 확대 시간창")]
    if any(zoom_parts(wo) for wo in tasks.WO_NO):
        handles.append(Line2D([0],[0],color=COL["PART"],marker="D",ls="None",label="교체 이력 · WO 완료 위치"))
    subtitle = (f"정비 경계 ±{pad_min}분: 온도 기록 없음 · 아래는 날짜상 직전 제공 LOT의 마지막 {nearest_minutes}분"
                if fallback else f"정비 경계 ±{pad_min}분 중 기록이 있는 시간창 · 겹치는 창은 합침 · 선: 1초 원본 · 실제 날짜·시각 유지")
    header(fig,title,subtitle,handles,ncol=5)
    compact_header(fig,.948,.916,title_size=24,sub_size=13,legend_size=12.5)
    axis_tag(fig,TAG_REAL)
    finish(fig,name)
    maintenance_zoom_checks.append({"FILE":name,"MODE":"직전 관측" if fallback else "정비 경계",
                                    "N_WINDOWS":len(windows),"WO":list(tasks.WO_NO),
                                    "ORIGINAL_CLOCK":True,"NO_INTERPOLATION":True})
    rows = maintenance_zoom_audit.loc[maintenance_zoom_audit.WO.isin(tasks.WO_NO)]
    table(rows)
    table(pd.DataFrame(maintenance_zoom_windows).loc[lambda d:d.그림.eq(name)])
    for task in tasks.itertuples():
        note(f"**작업 내용:** {task.WO_NO} · {task.MNT_TYPE} · {remark_text(task)}. "
             f"감지 {task.DETC_DT:%m/%d %H:%M} → 착수 {task.STRT_DT:%m/%d %H:%M} → 완료 {task.END_DT:%m/%d %H:%M}.")
        if zoom_parts(task.WO_NO):
            note("**같은 WO의 교체 이력:** " + "; ".join(zoom_parts(task.WO_NO)) + ". 다이아몬드 위치는 WO 완료 시각이며 실제 교체 실행 시각은 제공되지 않았다.")
    if fallback:
        task = tasks.iloc[0]
        last = shown_lots.iloc[-1]
        note(f"**이 확대의 범위:** {int(last.LOT_NO)}의 마지막 관측은 {last.END:%m/%d %H:%M:%S}이며 "
             f"감지까지 {(task.DETC_DT-last.END).total_seconds()/60:.2f}분, 착수까지 {(task.STRT_DT-last.END).total_seconds()/60:.2f}분 떨어져 있다. "
             "직전 제공 관측의 형태만 확인할 수 있고, 실제 BM 시점의 온도나 정비 뒤 회복을 이 그림으로 확인할 수 없다.")
    return windows

In [ ]:
def maintenance_stages(k,name):
    r=lot_meta.loc[k];tasks=linked_work(k)
    if not len(tasks):return
    cuts=sorted({r.START,r.END+pd.Timedelta(seconds=1),*[max(r.START,t.STRT_DT) for t in tasks.itertuples()],*[min(r.END+pd.Timedelta(seconds=1),t.END_DT) for t in tasks.itertuples()]})
    stages=[]
    for a,b in zip(cuts[:-1],cuts[1:]):
        active=tasks.loc[tasks.STRT_DT.lt(b)&tasks.END_DT.gt(a)]
        label='·'.join(active.MNT_TYPE) if len(active) else '작업 전' if b<=tasks.STRT_DT.min() else '전체 작업 완료 후'
        stages.append({'A':a,'B':b,'NAME':label,'COLOR':'#d9e8e5' if label=='전체 작업 완료 후' else '#f3e9d9' if len(active) else '#eef1f3'})
    fig=plt.figure(figsize=(28,25));gs=fig.add_gridspec(4,2,height_ratios=[.8,1,1,1.1],left=.10,right=.975,top=.87,bottom=.06,hspace=.70,wspace=.22)
    timeline=fig.add_subplot(gs[0,:]);timeline.plot([r.START,r.END],[0,0],color=LOT_COLOR[k],lw=8)
    timeline.set_yticks(range(1+len(tasks)),['온도 관측']+[f'{t.MNT_TYPE} · {t.WO_NO}\n{remark_text(t)}' for t in tasks.itertuples()]);timeline.set_ylim(len(tasks)+.45,-.65)
    for i,t in enumerate(tasks.itertuples(),1):
        timeline.plot([t.STRT_DT,t.END_DT],[i,i],color=COL[t.MNT_TYPE],lw=8)
        timeline.scatter(t.DETC_DT,i,marker='v',s=65,color=COL['DETECT'])
        if parts_for(t.WO_NO):timeline.scatter(t.END_DT,i,marker='D',s=65,color=COL['PART'])
    clock_axis(timeline);timeline.set_xlim(r.START,r.END);timeline.set_title('(a) 실제 작업의 시작·완료와 중복 구간',loc='left',fontsize=16,pad=12)
    full=raw_window(k,0,r.MINUTES)
    for row,(left,right) in enumerate([(HEAT,[f'OP-Z{z}' for z in range(3,7)]),(['CP-Z4'],['CP-Z4M'])],1):
        for col,ss in enumerate([left,right]):
            ax=fig.add_subplot(gs[row,col]);process_axes(ax,k,ss,0,r.MINUTES,raw_values=True,envelope=False)
            for st in stages:ax.axvspan(st['A'],st['B'],color=st['COLOR'],alpha=.45,lw=0,zorder=0)
            for t in tasks.itertuples():
                ax.axvline(t.STRT_DT,color=COL[t.MNT_TYPE],ls='--',lw=1.2);ax.axvline(t.END_DT,color=COL[t.MNT_TYPE],ls='-.',lw=1.2)
            ax.set_xlim(r.START,r.END);ax.set_title(f"({'bcde'[(row-1)*2+col]}) {' · '.join(ss)} · 실제 시각 원본",loc='left',fontsize=15,pad=12)
    for i,st in enumerate(stages):
        g=raw[k].loc[raw[k].MEAS_DT.ge(st['A'])&raw[k].MEAS_DT.lt(st['B'])]
        for s in SIGNALS:
            v=g[s].dropna();elapsed=(g.MEAS_DT-st['A']).dt.total_seconds()/60; ok=g[s].notna()
            slope=np.polyfit(elapsed[ok],g.loc[ok,s],1)[0] if ok.sum()>=60 else np.nan
            maintenance_stage_rows.append({'LOT':k,'단계':i,'구간':st['NAME'],'시작':st['A'],'끝':st['B'],'길이[분]':(st['B']-st['A']).total_seconds()/60,'센서':s,'유효 기록[개]':len(v),'중앙값':v.median(),'Q25':v.quantile(.25),'Q75':v.quantile(.75),'IQR':v.quantile(.75)-v.quantile(.25),'최소':v.min(),'최대':v.max(),'선형 변화율/분':slope})
    stat=pd.DataFrame(maintenance_stage_rows).loc[lambda d:d.LOT.eq(k)]
    for col,kind in enumerate(['TC','OP']):
        ax=fig.add_subplot(gs[3,col])
        for z in [3,4,5,6]:
            d=stat.loc[stat.센서.eq(f'{kind}-Z{z}')].sort_values('단계')
            ax.errorbar(d.단계+(z-4.5)*.06,d.중앙값,yerr=np.vstack([d.중앙값-d.Q25,d.Q75-d.중앙값]),fmt='o',color=ZONE_COLOR[z],capsize=5,label=f'{kind}-Z{z}')
        ax.set_xticks(range(len(stages)),[f"{s['NAME']}\n{s['A']:%H:%M}–{s['B']:%H:%M}" for s in stages],fontsize=11)
        ax.set_ylabel('온도[℃]' if kind=='TC' else '제어출력[%]');style(ax,'실제 작업 구간 · 점: 중앙값 · 막대: Q25–Q75')
        ax.set_title(f"({'fg'[col]}) {kind}-Z3~Z6 구간별 수준과 변동",loc='left',fontsize=15,pad=12)
    handles=[Line2D([0],[0],color=ZONE_COLOR[z],lw=2,label=f'Z{z}') for z in [3,4,5,6]]+[Patch(facecolor='#eef1f3',label='작업 전'),Patch(facecolor='#f3e9d9',label='작업 중'),Patch(facecolor='#d9e8e5',label='전체 작업 완료 후')]
    header(fig,f'{k} 정비 전·중·후 변화', '같은 LOT·실제 시각 · TBM/CBM 중복을 별도 단계로 표시 · 전후 수준·변동과 동시 신호',handles,ncol=7)
    compact_header(fig,.953,.925,title_size=24,sub_size=13,legend_size=12);axis_tag(fig,TAG_REAL);finish(fig,name)
    for stage_id, frame in stat.groupby('단계', sort=True):
        label = frame.iloc[0]
        note(f"**{label['구간']} — 22개 신호:** {label['시작']:%m/%d %H:%M}–{label['끝']:%m/%d %H:%M}. 단위는 각 센서의 축 기준이며 CP 두 값은 무차원이다.")
        table(frame.drop(columns=['LOT', '단계']))
    for s in ['TC-Z3','TC-Z4','OP-Z3','OP-Z4','CP-Z4']:
        d=stat.loc[stat.센서.eq(s)].sort_values('단계');pre=d.iloc[0];post=d.iloc[-1]
        note(f'**{s} 관측 차이:** 작업 전 중앙값 {pre.중앙값:.4g} → 전체 작업 완료 후 {post.중앙값:.4g}, 차이 {post.중앙값-pre.중앙값:+.4g}. IQR {pre.IQR:.4g} → {post.IQR:.4g}. 구간 길이 {pre["길이[분]"]:.1f}분 → {post["길이[분]"]:.1f}분이며, 초기 승온과 진행 중인 다른 작업을 함께 고려한다.')
    # 완료 전후 동일 30분창도 표로 제공한다. 완료 전에는 다른 WO 작업이 진행 중일 수 있다.
    around=[]
    for t in tasks.itertuples():
        for label,a,b in [('착수 전 30분',t.STRT_DT-pd.Timedelta(minutes=30),t.STRT_DT),('완료 후 30분',t.END_DT,t.END_DT+pd.Timedelta(minutes=30))]:
            g=raw[k].loc[raw[k].MEAS_DT.ge(a)&raw[k].MEAS_DT.lt(b)]
            others=cal_work.loc[cal_work.WO_NO.ne(t.WO_NO)&cal_work.STRT_DT.lt(b)&cal_work.END_DT.gt(a)]
            for s in SIGNALS:
                v=g[s].dropna();around.append({'WO':t.WO_NO,'구간':label,'시작':a,'끝':b,'다른 진행 작업':' / '.join(others.WO_NO),'센서':s,'유효 기록[개]':len(v),'중앙값':v.median(),'IQR':v.quantile(.75)-v.quantile(.25)})
    for (wo, window), frame in pd.DataFrame(around).groupby(['WO', '구간'], sort=False):
        note(f"**{wo} · {window} — 22개 신호:** 동일 길이의 창을 대조하며 다른 진행 작업 열을 함께 확인한다.")
        table(frame.drop(columns=['WO', '구간']))
    note('**해석 범위:** 이 표는 실제로 관측된 전후 차이다. 240117의 작업 전은 초기 승온이 포함되고, TBM 완료 뒤에도 CBM이 진행 중이다. 단계별 길이도 다르므로 수준·변동 차이를 정비 효과나 정상 회복으로 단정하지 않는다. 원본·전후 30분 표·다른 구역을 함께 확인한다.')

In [ ]:
# 과시효·냉각: TC-Z9·냉각은 원래 오르내리는 반복 파형이라, 같은 경과 분 차이와 피크를 맞춘 뒤 차이를 함께 본다 (기존 03과 같은 계산)
cycle_rows,phase,peak_dict,cycle_values,smooth_values=[],{},{},{},{}
CYCLIC=["TC-Z9","TC-OUT1","TC-OUT2"]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for s in CYCLIC:
        y=profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고"))
        # 원본을 채우지 않는다. 관측된 이웃 분 3개 이상의 사후 이동 중앙값으로만 피크를 찾는다.
        smooth=y.rolling(5,center=True,min_periods=3).median()
        cycle_values[k,s],smooth_values[k,s]=y,smooth
        peaks,_=find_peaks(smooth.to_numpy(),prominence=5,distance=80 if s=="TC-Z9" else 30)
        peak_dict[k,s]=peaks
        candidate_period=float(np.median(np.diff(peaks))) if len(peaks)>=3 else np.nan
        interval_iqr=float(np.quantile(np.diff(peaks),.75)-np.quantile(np.diff(peaks),.25)) if len(peaks)>=3 else np.nan
        consistent=bool(pd.notna(candidate_period) and interval_iqr/candidate_period<=.2)
        period=candidate_period if consistent else np.nan
        cycle_rows.append({"LOT":k,"GRADE":r.GRADE,"SENSOR":s,"N_PEAKS":len(peaks),"PEAK_INTERVAL_MEDIAN_MIN":candidate_period,"PERIOD_MIN":period,"PHASE_ELIGIBLE":consistent,
                          "PEAK_INTERVAL_IQR_MIN":interval_iqr,
                          "AMPLITUDE_Q95_Q05_C":y.quantile(.95)-y.quantile(.05),"CENTER_C":y.median(),
                          "ASSESSMENT":"피크 간격 일관·주기 비교 가능" if consistent else "피크 간격 불일치·보류" if len(peaks)>=3 else "검출 피크 3개 미만·주기 비교 보류"})
        if pd.notna(period):
            bins=np.floor((y.index.to_numpy()-peaks[0])%period).astype(int)
            phase[k,s]=pd.DataFrame({"PHASE_MIN":bins,"VALUE":y.to_numpy()}).groupby("PHASE_MIN").VALUE.median()
cycle_table=pd.DataFrame(cycle_rows)

phase_rows = []
phase_layout_audit = []

def phase_figure(grade, name):
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    valid_by_sensor={s:[k for k in ids if (k,s) in phase] for s in CYCLIC}
    comparable=[s for s in CYCLIC if len(valid_by_sensor[s]) >= 2]
    note(f"**그림 목적 — {grade} 주기 비교:** 위에서 세 센서의 전체 파형과 검출 피크를 확인한다. 아래는 두 LOT 이상이 피크 수·간격 조건을 충족하는 센서만, 피크 후 경과시간을 맞춰 비교한다.")
    fig=plt.figure(figsize=(24,12 if comparable else 7))
    gs=fig.add_gridspec(2 if comparable else 1,3,left=.09,right=.97,top=.77,
                        bottom=.12,hspace=.62,wspace=.28)
    for j,s in enumerate(CYCLIC):
        ax=fig.add_subplot(gs[0,j])
        for k in ids:
            line(ax,k,s)
            peaks=peak_dict[k,s]
            ax.scatter(peaks,smooth_values[k,s].loc[peaks],color=LOT_COLOR[k],s=22)
        ax.set_xlim(0,float(lots.loc[lots.GRADE.eq(grade),"MINUTES"].max()))
    if comparable:
        lower=gs[1,:].subgridspec(1,len(comparable),wspace=.28)
        for j,s in enumerate(comparable):
            ax=fig.add_subplot(lower[0,j])
            valid=valid_by_sensor[s]
            for k in valid:
                p=phase[k,s]
                ax.plot(p.index,p,color=LOT_COLOR[k],lw=1.8)
            ax.set_title(f"{PROCESS[s]} [{s}] · 피크를 맞춘 {len(valid)} LOT 비교",loc="left",pad=12)
            ax.set_ylabel("온도[℃]",labelpad=10)
            style(ax,"주기 안의 피크 후 경과시간[분]")
            for ix,k in enumerate(valid):
                for other in valid[ix+1:]:
                    pair=pd.concat([phase[k,s].rename("A"),phase[other,s].rename("B")],axis=1).dropna()
                    before_pair=pd.concat([cycle_values[k,s].rename("A"),
                                           cycle_values[other,s].rename("B")],axis=1).dropna()
                    before_mae=(before_pair.A-before_pair.B).abs().mean()
                    phase_rows.append({"ELAPSED_MAE_C":before_mae,"GRADE":grade,"SENSOR":s,"LOT_A":k,"LOT_B":other,"N_PHASE_MIN":len(pair),
                                       "PHASE_MAE_C":(pair.A-pair.B).abs().mean(),"PHASE_MEDIAN_DELTA_C":(pair.A-pair.B).median()})
    names=lambda s:s.replace('TC-','')
    excluded=cycle_table.loc[cycle_table.GRADE.eq(grade)&~cycle_table.PHASE_ELIGIBLE]
    details=[]
    for k,records in excluded.groupby('LOT',sort=False):
        reasons=[]
        for record in records.itertuples():
            reasons.append(f'{names(record.SENSOR)} 피크 {record.N_PEAKS}개' if record.N_PEAKS < 3
                           else f'{names(record.SENSOR)} 피크 간격 불일치')
        details.append(f"{k}({', '.join(reasons)})")
    comparison_text=('주기 비교: '+ '·'.join(names(s) for s in comparable)
                     + ('만 가능' if len(comparable) < len(CYCLIC) else ' 가능')) if comparable else '주기 비교 가능한 센서 없음'
    omitted_text=' / '.join(details)
    caption=(comparison_text + ' · 정렬 제외: ' + omitted_text + ' → 위쪽 전체 파형으로 확인'
             if details else comparison_text)
    fig.text(.09,.055,caption,ha='left',va='bottom',fontsize=12.5,color='#526d80')
    header(fig,f"{grade} 주기·위상 대조",
           "위: 전체 파형·검출 피크 / 아래: 두 LOT 이상 정렬 가능한 센서만 비교",
           lot_handles(ids)+[Line2D([0],[0],color="#34495e",marker="o",ls="None",label="각 LOT 색 점: 평활곡선 검출 피크"),QUALITY_HANDLE],ncol=4)
    axis_tag(fig,'TIME AXIS · 위: 관측 시작 후 / 아래: 주기 안 피크 후 [분]')
    assert len(fig.axes)==len(CYCLIC)+len(comparable)
    assert all(len(valid_by_sensor[s])>=2 for s in comparable)
    phase_layout_audit.append({'GRADE':grade,'FILE':name,'TOP_SENSORS':list(CYCLIC),
                               'LOWER_SENSORS':comparable,'VALID_LOTS':valid_by_sensor,
                               'N_AXES':len(fig.axes),'CAPTION':caption})
    finish(fig,name)
    pp=pd.DataFrame([z for z in phase_rows if z["GRADE"]==grade])
    if len(pp):
        for z in pp.itertuples():
            note(f"**위상 대조 결과:** [{z.SENSOR}] {z.LOT_A}·{z.LOT_B}의 동일 경과 분 평균 절대 차이 {z.ELAPSED_MAE_C:.3f}℃, 피크 정렬 뒤 {z.PHASE_MAE_C:.3f}℃다. 정렬 영향과 실제 중심·진폭 차이를 함께 검토한다.")
    for z in cycle_table.loc[cycle_table.GRADE.eq(grade)].itertuples():
        note(f"**파형 결과:** {z.LOT} [{z.SENSOR}] 중심 {z.CENTER_C:.3f}℃, Q95–Q05 폭 {z.AMPLITUDE_Q95_Q05_C:.3f}℃, 피크 {z.N_PEAKS}개, 주기 {f'{z.PERIOD_MIN:.1f}분' if pd.notna(z.PERIOD_MIN) else '판정 보류'}이다.")
    note(f"**비교 범위:** {comparison_text}. {omitted_text + '은(는) 정렬에서 제외하고 전체 파형으로 확인한다.' if details else ''} "
         "이번 정렬 기준은 검출 피크 3개 이상·피크 간격 IQR/중앙값 20% 이하다. 제외는 정상·이상 판정이 아니다.")
    table(cycle_table.loc[cycle_table.GRADE.eq(grade)])
    if len(pp):table(pp)


<a id="grade-sgcc"></a>

## SGCC — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240001"></a>

### 240001 전체 관측과 확인 위치

In [ ]:
full_lot(240001)
for win in lot_windows(240001):
 window_detail(240001,win)
lot_history(240001)

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

<a id="grade-spfh590"></a>

## SPFH590 — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240024"></a>

### 240024 전체 관측과 확인 위치

In [ ]:
full_lot(240024)
for win in lot_windows(240024):
 window_detail(240024,win)
lot_history(240024)

<a id="lot-240108"></a>

### 240108 전체 관측과 확인 위치

In [ ]:
full_lot(240108)
for win in lot_windows(240108):
 window_detail(240108,win)
lot_history(240108)

### SPFH590 같은 강종 전체 비교

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다.

구간별 강종 대조는 위의 각 LOT 확대 그림에 함께 배치했다. 이 절에서는 강종 전체 형상·차이·비교 가능한 시간·주기를 본다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPFH590",page,f"03_강종/SPFH590/{page}_전체비교.png")
grade_peer_note("SPFH590")
phase_figure("SPFH590","03_강종/SPFH590/Z9_냉각_주기위상.png")

<a id="grade-spce"></a>

## SPCE — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240037"></a>

### 240037 전체 관측과 확인 위치

In [ ]:
full_lot(240037)
for win in lot_windows(240037):
 window_detail(240037,win)
lot_history(240037)

<a id="lot-240121"></a>

### 240121 전체 관측과 확인 위치

In [ ]:
full_lot(240121)
for win in lot_windows(240121):
 window_detail(240121,win)
lot_history(240121)

<a id="lot-240133"></a>

### 240133 전체 관측과 확인 위치

In [ ]:
full_lot(240133)
for win in lot_windows(240133):
 window_detail(240133,win)
lot_history(240133)

<a id="mnt-1228"></a>

#### 감지·작업·교체와 관측 공백

감지 전후 변화는 확인할 수 있지만 작업 시작과 완료 이후의 온도 기록은 없다. 교체 효과는 확인할 수 없다고 표시한다.

In [ ]:
maintenance_zoom(["WO-2024-02105"],"04_정비전후/04_1228_CBM/01_240133_CBM_감지착수.png","12월 감지·작업·교체와 관측 범위")

### SPCE 같은 강종 전체 비교

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다.

구간별 강종 대조는 위의 각 LOT 확대 그림에 함께 배치했다. 이 절에서는 강종 전체 형상·차이·비교 가능한 시간·주기를 본다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPCE",page,f"03_강종/SPCE/{page}_전체비교.png")
grade_peer_note("SPCE")
phase_figure("SPCE","03_강종/SPCE/Z9_냉각_주기위상.png")

<a id="grade-spcen"></a>

## SPCEN — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240038"></a>

### 240038 전체 관측과 확인 위치

In [ ]:
full_lot(240038)
for win in lot_windows(240038):
 window_detail(240038,win)
lot_history(240038)

<a id="lot-240114"></a>

### 240114 전체 관측과 확인 위치

In [ ]:
full_lot(240114)
for win in lot_windows(240114):
 window_detail(240114,win)
lot_history(240114)

### SPCEN 같은 강종 전체 비교

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다.

구간별 강종 대조는 위의 각 LOT 확대 그림에 함께 배치했다. 이 절에서는 강종 전체 형상·차이·비교 가능한 시간·주기를 본다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPCEN",page,f"03_강종/SPCEN/{page}_전체비교.png")
grade_peer_note("SPCEN")
phase_figure("SPCEN","03_강종/SPCEN/Z9_냉각_주기위상.png")

<a id="grade-sphc"></a>

## SPHC — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240044"></a>

### 240044 전체 관측과 확인 위치

In [ ]:
full_lot(240044)
for win in lot_windows(240044):
 window_detail(240044,win)
lot_history(240044)

<a id="lot-240061"></a>

### 240061 전체 관측과 확인 위치

In [ ]:
full_lot(240061)
for win in lot_windows(240061):
 window_detail(240061,win)
lot_history(240061)

<a id="mnt-0609"></a>

#### 6월 정기수리 — 관측된 종료·수리·초기 상승을 연결

수리 구역 위치는 제공되지 않았다. Z3~Z6은 가열·균열의 동반 변화를 확인하는 선택이며, 전후 강종이 달라 정비 효과로 단정하지 않는다.

In [ ]:
saved_color=LOT_COLOR[240061];LOT_COLOR[240061]=GRADE_COLOR['SPHC']
fig,tasks06,ev06,b_win06=case_figure(plans.iloc[0],240058,240061,(0,110),"6월 정기수리 · 전체 이력 → 전후 원본")
finish(fig,"04_정비전후/01_0610-0614_6월_정기수리/01_240058_240061_수리전후.png")
table(work_table(tasks06));table(ev06[['EVT_DT','BEF_VAL','AFT_VAL']])
LOT_COLOR[240061]=saved_color
note(f"**관측 사실:** 240058 종료부는 여러 가열 구역이 함께 하강하고, 240061 초기는 함께 상승한다. 계획수리 중 제공 온도 기록은 없으며 첫 제공 관측은 계획 종료 {(lot_meta.loc[240061,'START']-plans.iloc[0].END_EXCL).total_seconds()/60:.0f}분 뒤다. 같은 WO 교체는 오일씰·O링 세트로 확인되지만, 정확한 교체 순간·구역별 수리 위치·재가동 소재 정보는 제공되지 않았다.")

#### 6월 정기수리 그래프 읽는 법

| 표시 | 뜻 |
|---|---|
| 맨 위 색 띠 | 강종 구간. 강종변경 시각부터 다음 변경까지의 강종이며, 띠 위 상자가 변경 시각이다 |
| 하늘색 배경 | CR1 계획수리 기간 |
| 온도 관측 막대 | 각 LOT의 온도 관측 시간 |
| 갈색·청록 막대 | 실제 TBM·CBM 작업 시간. 옆 글자는 REMARK·작업 길이·FAIL_MODE·교체 부품 |
| ▼ / ◆ | 정비 감지 시각(DETC_DT) / 같은 WO의 부품교체(WO 완료 시각 기준) |
| 빨간 상자 (b)(c) | 아래 (b)(c) 확대 그림이 보여주는 **위치**. 이상 표시가 아니다 |
| (b) 분홍 음영 | 종료 전환 구간. 가열 온도가 끝부분에서 내려가기 시작한 분부터 관측 끝까지 |
| (b)(c) 선 색 | 구역 Z3 파랑 · Z4 주황 · Z5 초록 · Z6 분홍. 왼쪽 온도, 오른쪽 출력. 위쪽 눈금은 관측 시작 후 경과 분 |

#### 6월 정기수리 그래프 해석

- **(a) 순서:** 06/09 05:26 240058(DP590) 관측 → 09:02 관측 끝 → 06/10~06/14 계획수리(연속소둔로 노내 정비) → 06/15 01:04 240061(SPHC) 관측. 첫 관측은 계획 종료 64분 뒤다.
- **(a) 수리 중:** 강종은 DP590 → SPCC(06/11 08:00) → DP590(06/13 10:34)으로 바뀌고 TBM 작업은 6건이다. 수리 기간에는 온도 기록이 없다. 감지 6건은 모두 감지 시각 ±10분에 온도 기록이 0행이라, 어떤 신호 때문에 감지됐는지 볼 수 없다.
- **(a) 부품교체:** WO-2024-00916 하나뿐이다(오일씰 ×2 수명도래, O링 세트 ×1 손상). REMARK는 비어 있다.
- **(b) 240058 끝부분:** 07:37부터 가열 Z3~Z6 네 구역이 함께 내려가, 마지막 5분은 중간보다 -42.6~-43.5℃ 낮다. 출력 Z4~Z6은 08:52까지 평소 수준(약 50~75%)을 유지하다가 그 뒤 네 구역이 함께 0 쪽으로 내려간다(마지막 5분 8~13%). **끄면 출력과 온도가 함께 내려가는 정상 정지**이며, 계획수리 시작 15시간 전이다.
- **(c) 240061 처음 110분:** 01:04~02:40(96분) 동안 가열 네 구역이 함께 약 +39~+42℃ 오른다. 관측 중 작업은 0건이다. **켜면 함께 올라가는 재가동 승온**이다.
- **(c) Z3:** 02:20~02:30에 Z3만 혼자 출렁인다(나머지 구역 평균보다 최대 -7.9℃). 출력도 Z3만 0~100% 사이를 오가고, Z4~Z6은 49~67%에 머문다. Z3 구역의 특징(03의 2절)이며, 승온 중이라 이상 후보로 보지 않는다.
- **그래서:** 이 그림의 큰 변화는 계획수리 일정에 맞춘 정지와 재가동이다. 전후 강종이 달라(DP590 → SPHC) 정비 효과는 판단하지 않는다.

<a id="lot-240069"></a>

### 240069 전체 관측과 확인 위치

In [ ]:
full_lot(240069)
for win in lot_windows(240069):
 window_detail(240069,win)
lot_history(240069)

<a id="mnt-0708"></a>

#### CP 변화 이후 BM 이력과 직전 제공 관측

감지 시각에 온도 자료가 없으면 직전 관측을 날짜를 보존해 따로 본다. 절연 저하는 정비 이력의 사유이며 온도 CSV의 직접 측정 항목이 아니다.

In [ ]:
maintenance_zoom(["WO-2024-01094"],"04_정비전후/02_0708_BM/01_240069_BM_직전관측.png","7월 BM 이력과 직전 제공 관측")
cp_event_flow(240069,"04_정비전후/02_0708_BM/02_240069_CP_작업선후.png")

<a id="lot-240091"></a>

### 240091 전체 관측과 확인 위치

In [ ]:
full_lot(240091)
for win in lot_windows(240091):
 window_detail(240091,win)
lot_history(240091)

### SPHC 같은 강종 전체 비교

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다.

구간별 강종 대조는 위의 각 LOT 확대 그림에 함께 배치했다. 이 절에서는 강종 전체 형상·차이·비교 가능한 시간·주기를 본다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPHC",page,f"03_강종/SPHC/{page}_전체비교.png")
grade_peer_note("SPHC")
phase_figure("SPHC","03_강종/SPHC/Z9_냉각_주기위상.png")

<a id="grade-dp590"></a>

## DP590 — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240058"></a>

### 240058 전체 관측과 확인 위치

In [ ]:
full_lot(240058)
for win in lot_windows(240058):
 window_detail(240058,win)
lot_history(240058)

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

<a id="grade-spcd"></a>

## SPCD — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240117"></a>

### 240117 전체 관측과 확인 위치

In [ ]:
full_lot(240117)
for win in lot_windows(240117):
 window_detail(240117,win)
lot_history(240117)

<a id="mnt-1104"></a>

#### 11월 정기수리 — 수리 전후와 겹친 실제 작업

In [ ]:
r117=lot_meta.loc[240117]
fig,tasks11,ev11,b_win11=case_figure(plans.iloc[1],240114,240117,(0,r117.MINUTES),"11월 정기수리 · 전체 이력 → 전후 원본")
finish(fig,"04_정비전후/03_1105-1113_11월_정기수리_TBM_CBM/01_240114_240117_수리전후.png");table(work_table(tasks11))
note(f"**관측 사실:** 첫 제공 관측은 계획수리 종료 {(r117.START-plans.iloc[1].END_EXCL).total_seconds()/3600:.1f}시간 뒤다. 관측 전의 운전 상태는 자료에 없다. 초기 상승은 첫 작업 착수보다 앞서므로 승온과 실제 작업을 구분한다.")

#### 11월 정기수리 그래프 읽는 법

| 표시 | 뜻 |
|---|---|
| 맨 위 색 띠 | 강종 구간. 강종변경 시각부터 다음 변경까지의 강종이며, 띠 위 상자가 변경 시각이다 |
| 하늘색 배경 | CR1 계획수리 기간 |
| 온도 관측 막대 | 각 LOT의 온도 관측 시간 |
| 갈색·청록 막대 | 실제 TBM·CBM 작업 시간. 옆 글자는 REMARK·작업 길이·FAIL_MODE·교체 부품 |
| ▼ / ◆ | 정비 감지 시각(DETC_DT) / 같은 WO의 부품교체(WO 완료 시각 기준) |
| 빨간 상자 (b)(c) | 아래 (b)(c) 확대 그림이 보여주는 **위치**. 이상 표시가 아니다 |
| (b) 분홍 음영 | 종료 전환 구간. 가열 온도가 끝부분에서 내려가기 시작한 분부터 관측 끝까지 |
| (b)(c) 선 색 | 구역 Z3 파랑 · Z4 주황 · Z5 초록 · Z6 분홍. 왼쪽 온도, 오른쪽 출력. 위쪽 눈금은 관측 시작 후 경과 분 |

#### 11월 정기수리 그래프 해석

- **(a) 순서:** 11/04 15:40 240114(SPCEN) 관측 → 20:59 관측 끝 → 11/05~11/10 계획수리(냉연 라인 전 설비 정기수리) → 11/12 15:00 CBM 감지(마모량 초과) → 11/13 07:08 240117(SPCD) 관측. 첫 관측은 계획 종료 55.1시간 뒤라, 그 사이 운전 상태는 자료에 없다.
- **(a) 수리 중:** 강종은 SPCEN → SPCE(11/07 14:08) → SPFH590(11/10 17:02)으로 바뀌고 TBM 작업은 5건(정기수리 실시·부속작업·점검)이다. 부품교체는 0건이다.
- **(a) 11/13:** 07:13 TBM 감지 → TBM 07:40~09:35(계획 점검 수행) → CBM 08:54~12:44(마모량 초과, FAIL_MODE WER). 두 작업은 41분 겹친다. **(c)에는 작업 시간 음영이 없으니, 시각은 (a)와 아래 표로 맞춰 본다.**
- **(b) 240114 끝부분:** 20:22부터 가열 네 구역이 함께 내려가, 마지막 5분은 중간보다 -41.6~-43.1℃ 낮다. 출력 Z4~Z6은 20:49까지 평소 수준을 유지하다가 함께 0 쪽으로 내려간다(마지막 5분 8~12%). **정상 정지**이며, 계획수리 시작 3시간 전이다. 다만 Z3은 온도가 내려가는 동안 출력이 20~100%를 오가며 온도도 계단처럼 출렁인다(Z3 구역의 특징).
- **(c) 240117 처음:** 07:08~08:37(89분) 동안 가열 네 구역이 함께 약 +37~+40℃ 오른다. **재가동 승온**이다. 승온은 첫 작업(07:40)보다 먼저 시작됐다.
- **(c) 승온 뒤:** Z4~Z6은 860℃ 근처에 붙어 있고 Z3만 위아래로 출렁인다(안정 구간 가운데 50% 폭: Z3 2.1℃, Z4~Z6 0.4~0.5℃). 출력도 Z3만 0~100%를 오가고(시간의 18%가 끝에 붙음), Z4~Z6은 45~69%에 머문다.
- **(c) 가장 볼 곳 · 12:55~13:05:** CBM 완료(12:44) 11분 뒤 OP-Z3가 0%가 됐는데(12:55~13:01), TC-Z3는 12:58에 870.3℃까지 오른다(작업 전 859.9℃). 같은 시간 Z4~Z6은 858~861℃에 그대로다. 이어 13:04에 CP-Z4가 이 LOT의 최저 0.367을 기록한다. 작업이 끝난 직후라는 시간 순서는 맞지만, CBM이 원인이라는 기록은 없다. 같은 Z3 모양이 작업 없는 다른 LOT에도 있다(03의 2절).
- **그래서:** 큰 하강·상승은 계획수리 일정에 맞춘 정지와 승온이다. 이 그림에서 따로 볼 곳은 12:55~13:05의 Z3이며, 03의 '240117 CBM 완료 뒤 Z3' 확대와 01의 정비 감지 원본 확대에서 이어서 본다.

<a id="mnt-1113"></a>

#### 정비 전·중·후 — 무엇이 얼마나 달라졌는가

작업 전 → TBM만 → TBM·CBM 중복 → CBM만 → 전체 작업 완료 후로 나눈다. 구간별 중앙값·IQR·범위·변화율과 전후 30분 표를 함께 확인한다. 작업 전의 승온과 진행 중인 다른 WO를 표시한다.

In [ ]:
maintenance_stages(240117,"04_정비전후/03_1105-1113_11월_정기수리_TBM_CBM/02_240117_정비전중후_변화.png")
cp_event_flow(240117,"04_정비전후/03_1105-1113_11월_정기수리_TBM_CBM/03_240117_CP_작업선후.png")

#### 감지·착수·완료 경계의 원본 확대

In [ ]:
maintenance_zoom(["WO-2024-01859","WO-2024-01854"],"04_정비전후/03_1105-1113_11월_정기수리_TBM_CBM/04_240117_TBM_CBM_경계확대.png","11월 TBM·CBM 경계 원본 확대")

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

## 전수 확인과 다음 연결

모든 R 후보는 전체 관측과 전수 표에 남긴다. 국소 후보는 겹친 W 창으로 원본을 확인하며 긴 수준 차이는 전체 파형과 같은 강종 비교에서 읽는다. [03 전체 통계·공통 패턴](05_all_data_timeseries_jinny_03.ipynb)은 이 관측을 수치로 다시 확인한다.

In [ ]:
assert set(overview_coverage)==set(lots.LOT_NO)
assert all(v==set(SIGNALS) for v in overview_coverage.values())
local=set(review_catalogue.loc[review_catalogue.LOCAL,'REVIEW_ID']);seen=set().union(*window_coverage.values())
assert local<=seen
table(review_catalogue);table(pd.DataFrame(window_audit));table(maintenance_zoom_audit)
table(pd.DataFrame(figure_checks))

# 파일명만으로 찾을 수 있게 그림 목록을 표로 보여준다(CSV로 따로 저장하지 않는다).
image_files = pd.DataFrame(image_name_manifest)
assert image_files['파일'].is_unique, '중복 그림 파일명'
assert not image_files['파일'].str.contains(r'-W\d+|_원본').any()
assert all(Path(p).name.count(str(k)) == 1 for k,p in zip(image_files.LOT,image_files['파일']))
assert all(len(str(OUT_DIR / p)) < 240 for p in image_files['파일']), 'Windows 경로 길이 확인'
table(image_files[['LOT','모드','시작','끝','확인 내용','파일']])
